# CodeQA Mini — cukup untuk membuktikan model bisa dijalankan

**Run All di CPU. Tidak perlu GPU, CodeBERT, Java, checkpoint besar, atau unduh dataset.**
Notebook membawa subset asli CodeQA Python: **256 train / 32 dev / 32 test**.
Satu-satunya dependensi tambahan adalah PyTorch; jika sudah terpasang, tidak perlu akses internet.

Model kecil: embedding + GRU encoder–decoder, dilatih dari nol. Pertanyaan dan kode menjadi input, jawaban menjadi output.
Ini **demo alur training dan inferensi**, bukan arsitektur CodeBERT atau reproduksi skor paper.
Jawaban mini model dapat salah; penurunan loss membuktikan training berjalan, bukan pemahaman kode yang baik.

Sumber: [paper](https://aclanthology.org/2021.findings-emnlp.223/),
[kode/dataset resmi](https://github.com/jadecxliu/CodeQA),
[dataset v1.0](https://drive.google.com/drive/folders/1i04sJNUHwMuDfMV2UfWeQG-Uv8MRw_qh).
Subset disampling dari split resmi masing-masing menggunakan seed 42. Kode dalam data tidak dieksekusi.

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from collections import Counter
import random, re, time, json, math, copy

START = time.perf_counter()
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(2, torch.get_num_threads()))
DEVICE = torch.device("cpu")
EPOCHS = 8
BATCH_SIZE = 16
MAX_SOURCE = 96
MAX_TARGET = 16
VOCAB_LIMIT = 2000
EMBED_SIZE = 64
HIDDEN_SIZE = 96
OUTPUT = Path.cwd() / "mini_codeqa_results"
OUTPUT.mkdir(exist_ok=True)
print("Device:", DEVICE, "| PyTorch:", torch.__version__)

Device: cpu | PyTorch: 2.14.1+cu130


## 1. Data sudah ada dalam notebook

Berkas dataset lokal tidak diperlukan. ID contoh dipertahankan untuk melacak contoh asal.
Vocabulary hanya dibuat dari train; dev dipakai memilih checkpoint; test dipakai setelah training.
Input panjang dipotong 96 token supaya demo tetap ringan.

In [2]:
MINI_DATA = {
 "train": [
  {
   "id": 41905,
   "code": "def add error e saved errors append e log '%- 70 s\\n' % e",
   "question": "What does the code append to the list of saved errors ?",
   "answer": "an error message"
  },
  {
   "id": 7296,
   "code": "def remove repo module repo cmd get cmd 'removerepo' repo rc stdout stderr module run command cmd check rc True return rc stdout stderr",
   "question": "What does the code remove ?",
   "answer": "the repo"
  },
  {
   "id": 1639,
   "code": "def stop name runas None service get service name path service['file path']return launchctl 'unload' path runas runas",
   "question": "What does the code stop ?",
   "answer": "a launchd service"
  },
  {
   "id": 48598,
   "code": "def remove vlan binding netid LOG debug 'remove vlan binding called' session db get session try binding session query network models v2 Vlan Binding filter by network id netid one session delete binding session flush return bindingexcept exc No Result Found pass",
   "question": "What does the code remove to network association ?",
   "answer": "a vlan"
  },
  {
   "id": 18024,
   "code": "def run all delay seconds 0 default scheduler run all delay seconds delay seconds",
   "question": "Who are scheduled to run or not ?",
   "answer": "they"
  },
  {
   "id": 16049,
   "code": "def fix value return value + 2 ** 32 if value < 0 else value",
   "question": "What does the code turn into positive numbers ?",
   "answer": "negative 32-bit numbers"
  },
  {
   "id": 14628,
   "code": "def jackknife resampling data n data shape[ 0 ]assert n > 0 u'datamustcontainatleastonemeasurement'resamples np empty [n n - 1 ] for i in range n resamples[i] np delete data i return resamples",
   "question": "Where does jackknife resampling perform ?",
   "answer": "on numpy arrays"
  },
  {
   "id": 9144,
   "code": "def nlmeans pipeline name ' Denoise' params {'patch radius' 1 'block radius' 5} inputnode pe Node niu Identity Interface fields ['in file' 'in mask'] name 'inputnode' outputnode pe Node niu Identity Interface fields ['out file'] name 'outputnode' nmask pe Node niu Function input names ['in file' 'in mask'] output names ['out file'] function bg mask name ' Noise Msk' nlmeans pe Node dipy Denoise **params name 'NL Means' wf pe Workflow name name wf connect [ inputnode nmask [ 'in file' 'in file' 'in mask' 'in mask' ] inputnode nlmeans [ 'in file' 'in file' 'in mask' 'in mask' ] nmask nlmeans [ 'out file' 'noise mask' ] nlmeans outputnode [ 'out file' 'out file' ] ] return wf",
   "question": "What performs nlmeans denoising example ?",
   "answer": "workflow"
  },
  {
   "id": 48265,
   "code": "def quote domain domain return urllib quote domain replace ' ' '% 2 E'",
   "question": "What have domain names tend ?",
   "answer": "to have"
  },
  {
   "id": 6717,
   "code": "@needs hostdef sudo command shell True pty True combine stderr None user None quiet False warn only False stdout None stderr None group None timeout None shell escape None capture buffer size None return run command command shell pty combine stderr sudo True user user if user else env sudo user group group quiet quiet warn only warn only stdout stdout stderr stderr timeout timeout shell escape shell escape capture buffer size capture buffer size",
   "question": "What does the code run on a remote host ?",
   "answer": "a shell command"
  },
  {
   "id": 44348,
   "code": "def after nearest workday dt return next workday nearest workday dt",
   "question": "For what purpose did nearest workday need ?",
   "answer": "for boxing day"
  },
  {
   "id": 48540,
   "code": "def stub urandom n random Data [random randrange 256 for n in xrange n ]return '' join map chr random Data",
   "question": "What provides in versions of python ?",
   "answer": "random data"
  },
  {
   "id": 35741,
   "code": "def absent name profile 'grafana' if isinstance profile string types profile salt ['config option'] profile ret {'result' None 'comment' None 'changes' None}datasource get datasource profile name if not datasource ret['result'] Trueret['comment'] ' Datasource{ 0 }alreadyabsent' format name return retrequests delete get url profile datasource['id'] headers get headers profile timeout profile get 'grafana timeout' 3 ret['result'] Trueret['comment'] ' Datasource{ 0 }wasdeleted' format name return ret",
   "question": "What does the code ensure ?",
   "answer": "that a data source is present"
  },
  {
   "id": 5697,
   "code": "def strip math s remove u'\\\\mathdefault' u'\\\\rm' u'\\\\cal' u'\\\\tt' u'\\\\it' u'\\\\' u'{' u'}' s s[ 1 -1 ]for r in remove s s replace r u'' return s",
   "question": "What does the code remove from mathtext ?",
   "answer": "latex formatting"
  },
  {
   "id": 38698,
   "code": "def matroska date to datetime date format re split u' [- ] ' u'%Y-%m-%d%H %M %S %f' while format try return datetime strptime date u'' join format except Value Error format format[ -2 ]return date",
   "question": "What converts to a python datetime object ?",
   "answer": "a date in matroskas date format"
  },
  {
   "id": 27651,
   "code": "def structured input Schema output Schema schema store None ignore body False if schema store is None schema store {}input Validator get Validator input Schema schema store output Validator get Validator output Schema schema store def deco original @wraps original @ remote logging@ logging@ serialize output Validator def load And Dispatch self request **route Arguments if request method in 'GET' 'DELETE' or ignore body objects {}else body request content read try objects loads body except Value Error raise DECODING ERRO Rerrors []for error in input Validator iter errors objects errors append error message if errors raise Invalid Request JSON errors errors schema input Schema objects update route Arguments return maybe Deferred original self **objects load And Dispatch input Schema input Schemaload And Dispatch output Schema output Schemareturn load And Dispatchreturn deco",
   "question": "What does the code decorate so that the request body is automatically decoded and the response body is automatically encoded ?",
   "answer": "a klein - style endpoint method"
  },
  {
   "id": 2082,
   "code": "def to locale language to lower False p language find '-' if p > 0 if to lower return language[ p] lower + ' ' + language[ p + 1 ] lower else return language[ p] lower + ' ' + language[ p + 1 ] upper else return language lower",
   "question": "What does the code turn into a locale name ?",
   "answer": "a language name"
  },
  {
   "id": 1952,
   "code": "def get categories app sa session app model context currentreturn sa session query app model Category filter app model Category table c deleted false order by app model Category table c name all",
   "question": "What does the code get from the database ?",
   "answer": "all categories"
  },
  {
   "id": 6140,
   "code": "def sensitive variables *variables def decorator func @functools wraps func def sensitive variables wrapper *args **kwargs if variables sensitive variables wrapper sensitive variables variableselse sensitive variables wrapper sensitive variables ' ALL 'return func *args **kwargs return sensitive variables wrapperreturn decorator",
   "question": "What used in the decorated function ?",
   "answer": "which variables"
  },
  {
   "id": 14328,
   "code": "def report error error exc info request None extra data None level None if HAS ROLLBAR and hasattr settings u'ROLLBAR' rollbar report exc info exc info request extra data extra data level level LOGGER error u' Handledexception%s %s' error class name force text error encode u'utf- 8 ' if sys argv[ 1 2] [u'test'] traceback print exc",
   "question": "When can this be used for store exceptions in error reporting solutions as rollbar ?",
   "answer": "while handling error gracefully and giving user cleaner message"
  },
  {
   "id": 15247,
   "code": "@requires segment infodef frame lister pl segment info full stack False maxframes 3 if full stack initial stack length 0frames segment info[u'pdb'] stackelse initial stack length segment info[u'initial stack length']frames segment info[u'pdb'] stack[initial stack length ]if len frames > maxframes frames frames[ - maxframes ]return {u'curframe' frame[ 0 ] u'initial stack length' initial stack length} {} for frame in frames",
   "question": "How do all frames list ?",
   "answer": "in segment_info format"
  },
  {
   "id": 33118,
   "code": "def homebrew bin ret salt ['cmd run'] 'brew--prefix' output loglevel 'trace' ret + '/bin/brew'return ret",
   "question": "What returns to the homebrew binary in the path ?",
   "answer": "the full path"
  },
  {
   "id": 39453,
   "code": "def parse file document file validate entry class entry keyword 'r' start position None end position None section end keywords extra args if start position document file seek start position else start position document file tell if section end keywords first keyword Noneline match KEYWORD LINE match stem util str tools to unicode document file readline if line match first keyword line match groups [0 ]document file seek start position if first keyword in section end keywords returnwhile end position is None or document file tell < end position desc lines ending keyword read until keywords entry keyword + section end keywords document file ignore first True end position end position include ending keyword True desc content bytes join '' desc lines if desc content yield entry class desc content validate *extra args if ending keyword in section end keywords breakelse break",
   "question": "What does the code read ?",
   "answer": "a range of the document_file containing some number of entry_class instances"
  },
  {
   "id": 1739,
   "code": "def relink or drop orphan translationprojects apps schema editor Project apps get model u'pootle project Project' TP apps get model u'pootle translationproject Translation Project' for proj key in set TP objects values list u'project id' flat True if not Project objects filter pk proj key exists for tp in TP objects filter project id proj key proj code tp pootle path split u'/' [2 ]projects Project objects filter code proj code if projects exists tp project projects first tp save else tp delete",
   "question": "How do tps relink ?",
   "answer": "with no project"
  },
  {
   "id": 36781,
   "code": "def ffmpeg extract audio inputfile output bitrate 3000 fps 44100 cmd [get setting 'FFMPEG BINARY' '-y' '-i' inputfile '-ab' '%dk' % bitrate '-ar' '%d' % fps output]subprocess call cmd",
   "question": "What does the code extract ?",
   "answer": "the sound from a video file"
  },
  {
   "id": 13031,
   "code": "def read denoiser mapping mapping fh denoiser mapping {}for i line in enumerate mapping fh if line '' continue centroid members line split ' ' denoiser mapping[centroid] members split return denoiser mapping",
   "question": "What did the code read ?",
   "answer": "the cluster mapping file handle mapping_fh : an open file handle to a cluster file"
  },
  {
   "id": 46925,
   "code": "def xframe options deny view func def wrapped view *args **kwargs resp view func *args **kwargs if resp get 'X- Frame- Options' is None resp['X- Frame- Options'] 'DENY'return respreturn wraps view func wrapped view",
   "question": "What did the x - frame - options http header set ?",
   "answer": "to deny"
  },
  {
   "id": 42590,
   "code": "def test uninstall setuptools develop install script data pkg path data packages join 'FS Pkg' script run 'python' 'setup py' 'develop' expect stderr True cwd pkg path script run 'python' 'setup py' 'install' expect stderr True cwd pkg path list result script pip 'list' '--format legacy' assert 'FS Pkg 0 1 dev 0 ' in list result stdout uninstall script pip 'uninstall' 'FS Pkg' '-y' assert any filename endswith ' egg' for filename in uninstall files deleted keys uninstall 2 script pip 'uninstall' 'FS Pkg' '-y' assert join script site packages 'FS Pkg egg-link' in uninstall 2 files deleted list uninstall 2 files deleted keys list result 2 script pip 'list' '--format legacy' assert 'FS Pkg' not in list result 2 stdout",
   "question": "What does the code try after setup ?",
   "answer": "uninstall"
  },
  {
   "id": 45962,
   "code": "def setup platform hass config add devices discovery info None add devices [ Zig Bee Switch hass Zig Bee Digital Out Config config ]",
   "question": "What does the code setup ?",
   "answer": "the zigbee switch platform"
  },
  {
   "id": 35713,
   "code": "def remove accents input str input str ustr input str nkfd form unicodedata normalize 'NFKD' input str return u'' join [c for c in nkfd form if not unicodedata combining c ]",
   "question": "What replaces accented latin letters suboptimal - but - better - than - nothing suboptimal - but - better - than - nothing ?",
   "answer": "by an ascii equivalent"
  },
  {
   "id": 27493,
   "code": "def full ct query action actor only None actstream registry check action actor query ct query action actor if action target is not None actstream registry check action target query ct query action target actor only if action action object is not None actstream registry check action action object query ct query action action object actor only return query",
   "question": "What matches an action ?",
   "answer": "a content type"
  },
  {
   "id": 14446,
   "code": "def flip g g def tr l return [ 1 - a for a in l]return meijerg tr g bm tr g bother tr g an tr g aother 1 / g argument",
   "question": "What does the code turn into one of inverse argument ?",
   "answer": "the g function"
  },
  {
   "id": 29439,
   "code": "def unpack files key list for key in key list underscore key find ' ' if underscore < 0 filename key if underscore < 0 else key[ 1 ] else filename '{basename} {ext}' format basename key[ underscore + 1 ] ext key[ underscore] clear metadata to file key filename",
   "question": "What does args clear ?",
   "answer": "the specified keys"
  },
  {
   "id": 38618,
   "code": "def update course enrollment username course id mode None is active None course key Course Key from string course id try user User objects get username username except User Does Not Exist msg u\" Notuserwithusername'{username}'found \" format username username log warn msg raise User Not Found Error msg try enrollment Course Enrollment objects get user user course id course key return update enrollment enrollment is active is active mode mode except Course Enrollment Does Not Exist return None",
   "question": "For what purpose does the code modify a course enrollment ?",
   "answer": "for a user"
  },
  {
   "id": 18231,
   "code": "def check code String filename reporter None builtins None if reporter is None reporter mod Reporter make Default Reporter try tree compile code String filename 'exec' ast Py CF ONLY AST except Syntax Error value sys exc info [1 ]msg value args[ 0 ] lineno offset text value lineno value offset value text if text is None reporter unexpected Error filename 'problemdecodingsource' else reporter syntax Error filename msg lineno offset text return 1except Exception reporter unexpected Error filename 'problemdecodingsource' return 1w checker Checker tree filename builtins builtins w messages sort key lambda m m lineno for warning in w messages reporter flake warning return len w messages",
   "question": "What does the code check ?",
   "answer": "the python source given by c{codestring } for flakes"
  },
  {
   "id": 53046,
   "code": "@pytest mark django dbdef test valid permissions for all modules for module in get modules url permissions set get permissions from urls module get urls module permissions set module get required permissions for permission in url permissions module permissions if module class in migrated permissions assert permission in migrated permissions[module class ] else assert get permission object from string permission",
   "question": "What does a module require ?",
   "answer": "permissions"
  },
  {
   "id": 425,
   "code": "def formstyle table inline form fields *args **kwargs def render row row id label widget comment hidden False class 'hide' if hidden else None row TR TD label class 'w 2 p fl' TD widget id row id class class if comment row append TD DIV class 'tooltip' title '%s %s' % label comment class 'w 2 p fc' return rowif args hidden kwargs get 'hidden' False return render row form fields args[ 0 ] args[ 1 ] hidden hidden else parent TABLE for row id label widget comment in fields row render row row id label widget comment parent append row return parent",
   "question": "Till when did by ifrc use old default eden form style ?",
   "answer": "still"
  },
  {
   "id": 49729,
   "code": "@pytest fixturedef en tutorial po member updated po directory settings english tutorial member store require store english tutorial settings POOTLE TRANSLATION DIRECTORY 'tutorial po' make member updates store member return store",
   "question": "What does the code require ?",
   "answer": "the /en / tutorial / tutorial"
  },
  {
   "id": 52810,
   "code": "def get Rotation By Split Line split Line return complex split Line[ 1 ] replace ' ' '' replace ' ' ''",
   "question": "What does the code get ?",
   "answer": "the complex rotation from the split gcode line"
  },
  {
   "id": 10463,
   "code": "def formstyle table form fields *args **kwargs def render row row id label widget comment hidden False row [] class 'hide' if hidden else None row append TR TD label class 'w 2 p fl' TD '' id row id + '1 ' class class row append TR widget TD comment class 'w 2 p fc' id row id class class return tuple row if args hidden kwargs get 'hidden' False return render row form fields args[ 0 ] args[ 1 ] hidden hidden else parent TABLE for row id label widget comment in fields rows render row row id label widget comment parent append rows[ 0 ] parent append rows[ 1 ] return parent",
   "question": "Till when did by ifrc use old default eden form style ?",
   "answer": "still"
  },
  {
   "id": 45753,
   "code": "def run epoch sess cost op ops reset num unrolls start timer sess run reset for in xrange num unrolls cost sess run [cost op] + ops [0 ]return timer - start cost",
   "question": "What does the code run ?",
   "answer": "one optimization epoch"
  },
  {
   "id": 27696,
   "code": "def update file finder getfinder fsr Carbon File FS Ref file file alias fsr FS New Alias Minimal return finder update file alias",
   "question": "What does the code update to match their on - disk representation ?",
   "answer": "the display of the specified object(s )"
  },
  {
   "id": 22298,
   "code": "def args from interpreter flags import subprocessreturn subprocess args from interpreter flags",
   "question": "What do command - line arguments reproduce ?",
   "answer": "the current settings in sys"
  },
  {
   "id": 18210,
   "code": "def base 64 url decode php style inp import base 64 padding factor 4 - len inp % 4 % 4 inp + ' ' * padding factor return base 64 b64 decode unicode inp translate dict zip map ord u'- ' u'+/'",
   "question": "What does php follow ?",
   "answer": "a slightly different protocol for base64 url decode"
  },
  {
   "id": 10189,
   "code": "def get programs for credentials user programs credentials certified programs []programs get programs user for program in programs for credential in programs credentials if program[u'uuid'] credential[u'credential'][u'program uuid'] program[u'credential url'] credential[u'certificate url']certified programs append program return certified programs",
   "question": "What did the code give ?",
   "answer": "a user and an iterable of credentials"
  },
  {
   "id": 14110,
   "code": "def rel posix to abs local host path environ None if environ is None environ os environif path startswith '/' path path[ 1 ]root path for host host environ return os path join root *path split '/'",
   "question": "What does the code prepend ?",
   "answer": "the tmp directory the hosts files are in"
  },
  {
   "id": 50036,
   "code": "def create cluster **kwargs cluster name kwargs get 'name' cluster spec kwargs get 'cluster spec' datacenter kwargs get 'datacenter' if cluster name is None raise Value Error ' Missingvalueforname ' if datacenter is None raise Value Error ' Missingvaluefordatacenter ' if cluster spec is None cluster spec vim cluster Config Spec Ex host folder datacenter host Foldercluster host folder Create Cluster Ex name cluster name spec cluster spec return cluster",
   "question": "Where do a cluster create ?",
   "answer": "in vcenter"
  },
  {
   "id": 22059,
   "code": "def render placeholder context placeholder name default layout None template name None global type False renderer Placeholder Renderer context placeholder name default layout default layout template name template name global type global type return renderer render",
   "question": "What does the code render in a given context ?",
   "answer": "a placeholder"
  },
  {
   "id": 6698,
   "code": "def send beta role email action user email params if action 'add' email params['message'] 'add beta tester'email params['email address'] user emailemail params['full name'] user profile nameelif action 'remove' email params['message'] 'remove beta tester'email params['email address'] user emailemail params['full name'] user profile nameelse raise Value Error \" Unexpectedactionreceived'{}'-expected'add'or'remove'\" format action send mail to student user email email params language get user email language user",
   "question": "What does the code send to a user added or removed as a beta tester ?",
   "answer": "an email"
  },
  {
   "id": 6078,
   "code": "def status name sig None jail None if sig return bool salt ['status pid'] sig cmd '{ 0 }{ 1 }onestatus' format cmd jail name return not salt ['cmd retcode'] cmd python shell False ignore retcode True",
   "question": "What does the code return ?",
   "answer": "the status for a service"
  },
  {
   "id": 24898,
   "code": "@contextlib contextmanagerdef silence old stdout sys stdoutold stderr sys stderrsys stdout Dummy File sys stderr Dummy File exception occurred Falsetry yield except exception occurred Truesys stdout old stdoutsys stderr old stderrraiseif not exception occurred sys stdout old stdoutsys stderr old stderr",
   "question": "What does a context manager silence ?",
   "answer": "sys"
  },
  {
   "id": 6338,
   "code": "def test warning on non existant path File Link fl display File Link 'example txt' nt assert true fl repr html startswith ' Path <tt>example txt</tt> '",
   "question": "What returns a warning ?",
   "answer": "calling _ repr_html _ on non - existant files"
  },
  {
   "id": 23526,
   "code": "def test parse factory source count fname os path basename source def test parse self trees Phylo XMLIO parse source self assert Equal len list trees count test parse doc ' Parsethephylogeniesin%s ' % fname return test parse",
   "question": "What does the code generate ?",
   "answer": "a test method for parse()ing the given source"
  },
  {
   "id": 55541,
   "code": "def markdown column params yield tables Move None 'effect' yield tables Move None 'short effect' for cls in tables mapped classes for translation cls in cls translation classes for column in translation cls table c if column info get 'string getter' markdown Markdown String yield cls translation cls column name",
   "question": "What does the code check all markdown values scan ?",
   "answer": "the database schema for markdown columns"
  },
  {
   "id": 22541,
   "code": "def remove fname try os remove fname except OS Error pass",
   "question": "What does the code remove ?",
   "answer": "a file"
  },
  {
   "id": 39565,
   "code": "@contextmanagerdef mock registered transformers transformers with patch 'openedx core lib block structure transformer registry Transformer Registry get registered transformers' as mock available transforms mock available transforms return value {transformer for transformer in transformers} yield",
   "question": "What returns the given transformers ?",
   "answer": "the transformer registry"
  },
  {
   "id": 17335,
   "code": "def inv a return solve a eye a shape[ 0 ] chunks a chunks[ 0 ][ 0 ]",
   "question": "How does the code compute the inverse of a matrix ?",
   "answer": "with lu decomposition and forward / backward substitutions"
  },
  {
   "id": 52895,
   "code": "def get Priority element try return get Priority Strict element except Value Error return Max",
   "question": "What does the code get ?",
   "answer": "the priority of this element returns max if no priority is specified or the priority value is invalid"
  },
  {
   "id": 2847,
   "code": "def morsel to cookie morsel c create cookie name morsel key value morsel value version morsel['version'] or 0 port None port specified False domain morsel['domain'] domain specified bool morsel['domain'] domain initial dot morsel['domain'] startswith ' ' path morsel['path'] path specified bool morsel['path'] secure bool morsel['secure'] expires morsel['max-age'] or morsel['expires'] discard False comment morsel['comment'] comment url bool morsel['comment'] rest {' Http Only' morsel['httponly']} rfc 2109 False return c",
   "question": "What do a cookie contain ?",
   "answer": "the one k / v pair"
  },
  {
   "id": 47823,
   "code": "def config dev None **kwargs if dev is None spath fspath else spath bcpath dev updates dict [ key val for key val in kwargs items if not key startswith ' ' ] if updates endres 0for key val in updates items endres + sysfs attr [spath key] val 'warn' ' Failedtoupdate{ 0 }with{ 1 }' format os path join spath key val return endres > 0 else result {}data sysfs parse spath config True internals True options True for key in 'other ro' 'inter ro' if key in data del data[key]for key in data keys result update data[key] return result",
   "question": "What show config ?",
   "answer": "config"
  },
  {
   "id": 30108,
   "code": "def plugin return Select Bracket",
   "question": "What does the code make ?",
   "answer": "plugin available"
  },
  {
   "id": 35142,
   "code": "@pytest fixture autouse True def fast wait monkeypatch old sleep gevent sleepold joinall gevent joinallold killall gevent killalldef fast wait tm return old sleep 0 1 def fast joinall *args **kwargs if 'timeout' in kwargs kwargs['timeout'] 0 1return old joinall *args **kwargs def fast killall *args **kwargs if 'timeout' in kwargs kwargs['timeout'] 0 1return old killall *args **kwargs monkeypatch setattr gevent 'sleep' fast wait monkeypatch setattr gevent 'joinall' fast joinall monkeypatch setattr gevent 'killall' fast killall",
   "question": "What take timeouts to wait briefly ?",
   "answer": "gevent calls"
  },
  {
   "id": 8180,
   "code": "def update last login sender user **kwargs user last login timezone now user save",
   "question": "What updates the last_login date for the user logging in ?",
   "answer": "a signal receiver"
  },
  {
   "id": 24807,
   "code": "def list prefix None bin env None user None cwd None packages {}if prefix is None or 'pip' startswith prefix packages['pip'] version bin env for line in freeze bin env bin env user user cwd cwd if line startswith '-f' or line startswith '#' continueelif line startswith '-ehg+nottrust' continueelif line startswith '-e' line line split '-e' [1 ] version name line split '#egg ' elif len line split ' ' > 2 name line split ' ' [0 ]version line split ' ' [1 ]else logger error \" Can'tparseline'{ 0 }'\" format line continueif prefix if name lower startswith prefix lower packages[name] version else packages[name] version return packages",
   "question": "Where do prefix exist ?",
   "answer": "in the list of packages installed"
  },
  {
   "id": 5164,
   "code": "def split path path path path lstrip '/' first rest path partition '/' lang first lower if lang in settings LANGUAGE URL MAP return settings LANGUAGE URL MAP[lang] rest elif get non supported lang is not None return get non supported lang rest else supported find supported first if supported return supported[ 0 ] rest else return '' path",
   "question": "What did the code split ?",
   "answer": "the requested path into"
  },
  {
   "id": 36178,
   "code": "def load list setting settings name value settings get name if not value value []if isinstance value str cls value [value]return value",
   "question": "What change settings that should be lists to just individual strings accidentally sometimes ?",
   "answer": "users"
  },
  {
   "id": 19213,
   "code": "def wait io timeout None timeout exc NONE if io callback is not None raise Concurrent Object Use Error ' Thissocketisalreadyusedbyanothergreenlet %r' % io callback if timeout is not None timeout exc timeout exc if timeout exc is not NONE else timeout error 'timedout' timeout Timeout start new timeout timeout exc try return get hub wait io finally if timeout is not None timeout cancel",
   "question": "What does the code block until * io * is ready ?",
   "answer": "the current greenlet"
  },
  {
   "id": 54355,
   "code": "def tamper payload **kwargs headers kwargs get 'headers' {} headers['X- Forwarded- For'] random IP return payload",
   "question": "For what purpose does the code append a fake http header x - forwarded - ?",
   "answer": "for to bypass waf protection"
  },
  {
   "id": 41198,
   "code": "def update symlinks config unused plugins cert manager update live symlinks config",
   "question": "What makes symlinks point to the correct archive directory ?",
   "answer": "the certificate file family symlinks"
  },
  {
   "id": 40535,
   "code": "def get url prefix return getattr local 'prefix' None",
   "question": "What does the code get ?",
   "answer": "the prefix for the current thread"
  },
  {
   "id": 23700,
   "code": "def low data queue False **kwargs conflict check queue queue kwargs if conflict is not None return conflicttry st salt state State opts proxy proxy except Name Error st salt state State opts err st verify data data if err context ['retcode'] 1return errret st call data if isinstance ret list context ['retcode'] 1if salt utils check state result ret context ['retcode'] 2return ret",
   "question": "What be needed in everyday usage ?",
   "answer": "the code execute a single low data call"
  },
  {
   "id": 37837,
   "code": "def get Geometry Output By Arguments arguments element Node return get Geometry Output None element Node",
   "question": "What does the code get from attribute dictionary by arguments ?",
   "answer": "vector3 vertexes"
  },
  {
   "id": 12601,
   "code": "def plot Outputs Over Time vectors bu Vectors None title ' On-times' import pylabpylab ion pylab figure im Data vectors transpose if bu Vectors is not None assert bu Vectors shape vectors shape im Data im Data copy im Data[bu Vectors transpose astype 'bool' ] 2pylab imshow im Data aspect 'auto' cmap pylab cm gray r interpolation 'nearest' pylab title title",
   "question": "What does the code generate ?",
   "answer": "a figure that shows each output over time"
  },
  {
   "id": 46174,
   "code": "def get easy ec 2 config file None cache False cfg get config config file cache return cfg get easy ec 2",
   "question": "What does factory for easyec2 class attempt ?",
   "answer": "to load aws credentials from the starcluster config file"
  },
  {
   "id": 4558,
   "code": "def is prime number cache True if number < 2 return Falseglobal primes cache list primes cache boolif cache and len primes cache list 0 primes cache list primes cache bool eratosthenes CACHE LIMIT return boolean True for prime in primes cache list primes cache bool[prime] Trueif number < len primes cache bool return primes cache bool[number]sqrt number sqrt number for prime in primes cache list if prime > sqrt number return Trueif number % prime 0 return Falseto check 2if len primes cache list > 0 to check primes cache list[ -1 ] + 1 while to check < sqrt number if number % to check 0 return Falseto check + 1return True",
   "question": "What does the code take ?",
   "answer": "number"
  },
  {
   "id": 3003,
   "code": "def get Voronoi Loop By Points inside loop outsides for outside in outsides loop get Voronoi Loop By Point inside loop outside return loop",
   "question": "What does the code get ?",
   "answer": "voronoi loop enclosing the inside"
  },
  {
   "id": 43336,
   "code": "def test cwd with absolute paths existing '/some/existing/path'additional 'another'absolute '/absolute/path'with settings cwd existing with cd absolute eq env cwd absolute with cd additional eq env cwd existing + '/' + additional",
   "question": "What should cd append if non - absolute ?",
   "answer": "arg"
  },
  {
   "id": 14935,
   "code": "def get interface addresses try output subprocess check output 'ifconfig' except subprocess Called Process Error return [' 127 0 0 1'] [] v4 re findall 'inet addr ? [^]* ' output v6 re findall 'inet 6 addr ? [^]* ' output v4 [e[ 1 ] for e in v4 ]v 6 [e[ 1 ] for e in v6 ]return v4 v6",
   "question": "What does the code get ?",
   "answer": "all ip addresses assigned to interfaces"
  },
  {
   "id": 50663,
   "code": "def find interfaces *args brs os dispatch 'brshow' if not brs return Noneiflist {}for iface in args for br in brs try if iface in brs[br]['interfaces'] iflist[iface] brexcept Exception passreturn iflist",
   "question": "Where are the interfaces are bond to cli ?",
   "answer": "the bridge"
  },
  {
   "id": 18965,
   "code": "def main parser argparse Argument Parser description ' Ansibledynamicinventory' parser add argument '--list' help ' Ansibleinventoryofallofthegroups' action 'store true' dest 'list inventory' parser add argument '--host' help ' Ansibleinventoryofaparticularhost' action 'store' dest 'ansible host' type str cli args parser parse args list inventory cli args list inventoryansible host cli args ansible hostif list inventory output list inventory ANSIBLE INV if ansible host find host ansible host HOST VARS",
   "question": "How did data structures define ?",
   "answer": "statically"
  },
  {
   "id": 5229,
   "code": "def get transport and path from url url config None **kwargs parsed urlparse urlparse url if parsed scheme 'git' return TCP Git Client from parsedurl parsed **kwargs parsed path elif parsed scheme in 'git+ssh' 'ssh' path parsed pathif path startswith '/' path parsed path[ 1 ]return SSH Git Client from parsedurl parsed **kwargs path elif parsed scheme in 'http' 'https' return Http Git Client from parsedurl parsed config config **kwargs parsed path elif parsed scheme 'file' return default local git client cls from parsedurl parsed **kwargs parsed path raise Value Error \"unknownscheme'%s'\" % parsed scheme",
   "question": "What does the code obtain from a url ?",
   "answer": "a git client"
  },
  {
   "id": 56054,
   "code": "def simplify units pass",
   "question": "Where do units appear ?",
   "answer": "in both numerator and denominator"
  },
  {
   "id": 15256,
   "code": "def add bgp error metadata code sub code def desc 'unknown' if EXCEPTION REGISTRY get code sub code is not None raise Value Error 'BGPS Exceptionwithcode%dandsub-code%dalreadydefined ' % code sub code def decorator subclass ' Setsclassconstantsforexceptioncodeandsub-code \\n\\n Ifgivenclassissub-classof BGPS Exceptionwesetsclassconstants \\n'if issubclass subclass BGPS Exception EXCEPTION REGISTRY[ code sub code ] subclasssubclass CODE codesubclass SUB CODE sub codesubclass DEF DESC def descreturn subclassreturn decorator",
   "question": "What want to set exception class meta - data ?",
   "answer": "all exceptions"
  },
  {
   "id": 6619,
   "code": "def positional max pos args ndb debug 'SKIP'def positional decorator wrapped if not DEBUG return wrapped ndb debug 'SKIP'@wrapping wrapped def positional wrapper *args **kwds ndb debug 'SKIP'if len args > max pos args plural s ''if max pos args 1 plural s 's'raise Type Error '%s takesatmost%dpositionalargument%s %dgiven ' % wrapped name max pos args plural s len args return wrapped *args **kwds return positional wrapperreturn positional decorator",
   "question": "What do a decorator declare ?",
   "answer": "that only the first n arguments may be positional"
  },
  {
   "id": 24911,
   "code": "def resource delete context data dict model context['model']id get or bust data dict 'id' entity model Resource get id if entity is None raise Not Found check access 'resource delete' context data dict package id entity get package id pkg dict get action 'package show' context {'id' package id} for plugin in plugins Plugin Implementations plugins I Resource Controller plugin before delete context data dict pkg dict get 'resources' [] if pkg dict get 'resources' pkg dict['resources'] [r for r in pkg dict['resources'] if not r['id'] id ]try pkg dict get action 'package update' context pkg dict except Validation Error as e errors e error dict['resources'][ -1 ]raise Validation Error errors for plugin in plugins Plugin Implementations plugins I Resource Controller plugin after delete context pkg dict get 'resources' [] model repo commit",
   "question": "What does the code delete from a dataset ?",
   "answer": "a resource"
  },
  {
   "id": 18217,
   "code": "def create spline y yp x h from scipy interpolate import P Poly n m y shapec np empty 4 n m - 1 dtype y dtype slope y[ 1 ] - y[ -1 ] / h t yp[ -1 ] + yp[ 1 ] - 2 * slope / h c[ 0 ] t / h c[ 1 ] slope - yp[ -1 ] / h - t c[ 2 ] yp[ -1 ]c[ 3 ] y[ -1 ]c np rollaxis c 1 return P Poly c x extrapolate True axis 1",
   "question": "What does the code create ?",
   "answer": "a cubic spline given values and derivatives"
  },
  {
   "id": 29714,
   "code": "def disable layer r **attr if r component name 'layer entity' session error T ' Incorrectparameters' redirect URL args [r id 'layer entity'] ltable s3 db gis layer configquery ltable config id r id & ltable layer id r component id db query update enabled False session confirmation T ' Layerhasbeen Disabled' redirect URL args [r id 'layer entity']",
   "question": "What designed to be a custom method called by an action button in config / layer_entity @todo ?",
   "answer": "a layer"
  },
  {
   "id": 41660,
   "code": "def format plugin info table info table column lengths info table insert 0 separator ' ' column lengths info table insert 1 ' Plugin' ' Description' info table insert 2 separator '-' column lengths info table append separator ' ' column lengths",
   "question": "What adds to plugin info table ?",
   "answer": "separators and column titles"
  },
  {
   "id": 54664,
   "code": "@register filter is safe True def intcomma value use l10 n True if settings USE L10 N and use l10 n try if not isinstance value float value int value except Type Error Value Error return intcomma value False else return number format value force grouping True orig force text value new re sub u'^ -?\\\\d+ \\\\d{ 3 } ' u'\\\\g< 1 > \\\\g< 2 >' orig if orig new return newelse return intcomma new use l10 n",
   "question": "What is containing commas every three digits ?",
   "answer": "a string"
  },
  {
   "id": 23909,
   "code": "def present name database duration '7 d' replication 1 default False **client args ret {'name' name 'changes' {} 'result' True 'comment' 'retentionpolicy{ 0 }isalreadypresent' format name }if not salt ['influxdb retention policy exists'] name name database database **client args if opts ['test'] ret['result'] Noneret['comment'] '{ 0 }isabsentandwillbecreated' format name return retif salt ['influxdb create retention policy'] database name duration replication default **client args ret['comment'] 'retentionpolicy{ 0 }hasbeencreated' format name ret['changes'][name] ' Present'return retelse ret['comment'] ' Failedtocreateretentionpolicy{ 0 }' format name ret['result'] Falsereturn retreturn ret",
   "question": "What does the code ensure ?",
   "answer": "that given retention policy is present"
  },
  {
   "id": 10659,
   "code": "def feature enabled feature config return utils grep '^%s y' % feature config",
   "question": "What does the code verify ?",
   "answer": "whether a given kernel option is enabled"
  },
  {
   "id": 24260,
   "code": "def get ipython from I Python core interactiveshell import Interactive Shellif Interactive Shell initialized return Interactive Shell instance",
   "question": "What does the code get ?",
   "answer": "the global interactiveshell instance"
  },
  {
   "id": 23283,
   "code": "def check missing generic role folder base generic role names provider constants monkeypatch assert folder base is not None role map constants['gmail role map'] if provider 'gmail' else constants['imap role map'] keys to remove []for folder name in role map if role map[folder name] role keys to remove append folder name for key in keys to remove del role map[key]for role alias in generic role names folders folder base + [ '\\\\ Has No Children' '/' role alias ] client patch gmail client monkeypatch folders if provider 'gmail' else patch generic client monkeypatch folders raw folders client folders role map[role alias] rolegeneric folder checks raw folders role map client provider del role map[role alias]",
   "question": "What label every folder in generic_role_names as input role role : the role that the generic_role_names should be assigned folder_base : generic list of folders ?",
   "answer": "clients"
  },
  {
   "id": 13730,
   "code": "def margulis gabber galil graph n create using None if create using is None create using nx Multi Graph elif create using is directed or not create using is multigraph msg '`create using`mustbeanundirectedmultigraph 'raise nx Network X Error msg G create using G clear for x y in itertools product range n repeat 2 for u v in x + 2 * y % n y x + 2 * y + 1 % n y x y + 2 * x % n x y + 2 * x + 1 % n G add edge x y u v G graph['name'] 'margulis gabber galil graph {0 } ' format n return G",
   "question": "What returns on n^2 nodes ?",
   "answer": "the margulis - gabber - galil undirected multigraph"
  },
  {
   "id": 43920,
   "code": "def decrypt cypher key return gluechops cypher key['d'] key['p'] * key['q'] decrypt int",
   "question": "What decrypts a cypher ?",
   "answer": "a cypher"
  },
  {
   "id": 17496,
   "code": "def run tests test labels verbosity 1 interactive True extra tests None **kwargs extra tests extra tests or [] app labels getattr settings 'TEST APPS' test labels return django test runner app labels verbosity verbosity interactive interactive extra tests extra tests **kwargs",
   "question": "What runs tests for the apps listed in settings only ?",
   "answer": "test runner"
  },
  {
   "id": 45994,
   "code": "def erfa check ira idec astrom cra cdec erfa atciq ira idec 0 0 0 0 astrom az zen ha odec ora erfa atioq cra cdec astrom alt np pi / 2 - zen cra 2 cdec 2 erfa atoiq u'A' az zen astrom ira 2 idec 2 erfa aticq cra 2 cdec 2 astrom dct locals del dct[u'astrom']return dct",
   "question": "What does this function do ?",
   "answer": "the same thing the astropy layer is supposed to do"
  },
  {
   "id": 44796,
   "code": "def main cisco file 'cisco ipsec txt'cisco cfg Cisco Conf Parse cisco file crypto maps cisco cfg find objects '^cryptomap CRYPTO' for c map in crypto maps printprint c map textfor child in c map children print child textprint",
   "question": "What does the code find ?",
   "answer": "all of the crypto map entries in the file"
  },
  {
   "id": 42469,
   "code": "def dict to etree d parent for k v in d items if v is None etree Sub Element parent k elif isinstance v six string types etree Sub Element parent k text velif isinstance v dict or isinstance v odict child etree Sub Element parent k dict to etree v child elif not isinstance v collections Sized etree Sub Element parent k text str v elif len v 0 etree Sub Element parent k else for e in v child etree Sub Element parent k if isinstance e dict or isinstance e odict dict to etree e child else child text str e",
   "question": "What does the code take ?",
   "answer": "a the dict whose value is either none or an instance of dict"
  },
  {
   "id": 4679,
   "code": "def source to unicode txt errors 'replace' skip encoding cookie True if isinstance txt str return txtif isinstance txt bytes buf io Bytes IO txt else buf txttry encoding detect encoding buf readline except Syntax Error encoding 'ascii'buf seek 0 text io Text IO Wrapper buf encoding errors errors line buffering True text mode 'r'if skip encoding cookie return u'' join strip encoding cookie text else return text read",
   "question": "What converts to unicode with python source code ?",
   "answer": "a bytes string"
  },
  {
   "id": 39920,
   "code": "def audio data generator buff stop Falsewhile not stop data [buff get ]while True try data append buff get block False except queue Empty breakif None in data stop Truedata remove None yield '' join data",
   "question": "What does a generator yield in the given buffer ?",
   "answer": "all available data"
  },
  {
   "id": 41613,
   "code": "def login required func login url None redirect REDIRECT FIELD NAME only active True if only active def redirect func user return not user is authenticated and user is active else def redirect func user return not user is authenticated return user access decorator redirect func redirect field redirect redirect url func lambda login url func",
   "question": "What does the code require ?",
   "answer": "that the user is logged in"
  },
  {
   "id": 11215,
   "code": "def get scheduler events event scheduler events frappe cache get value u'scheduler events' if not scheduler events scheduler events frappe get hooks u'scheduler events' integration events get integration service events for key handlers in integration events items scheduler events setdefault key [] extend handlers frappe cache set value u'scheduler events' scheduler events return scheduler events get event or []",
   "question": "What does the code get from hooks and integrations ?",
   "answer": "scheduler events"
  },
  {
   "id": 35005,
   "code": "def module disabled module disable module module reload service 'apache 2 '",
   "question": "What does the code require ?",
   "answer": "an apache module to be disabled"
  },
  {
   "id": 47784,
   "code": "def is library missing name path module name rsplit u' ' 1 try package import module path return not module has submodule package module except Import Error return is library missing path",
   "question": "Can library that failed to load be found under any templatetags directory ?",
   "answer": "No"
  },
  {
   "id": 16043,
   "code": "def degree centrality G nodes top set nodes bottom set G - top s 1 0 / len bottom centrality dict n d * s for n d in G degree top s 1 0 / len top centrality update dict n d * s for n d in G degree bottom return centrality",
   "question": "What does the code compute ?",
   "answer": "the degree centrality for nodes in a bipartite network"
  },
  {
   "id": 10708,
   "code": "def draw to notebook layers **kwargs from I Python display import Imagelayers layers get all layers if hasattr layers 'get all layers' else layers dot make pydot graph layers **kwargs return Image dot create png",
   "question": "Where does the code draw a network diagram ?",
   "answer": "in an ipython notebook"
  },
  {
   "id": 30294,
   "code": "def patch multiple target spec None create False spec set None autospec None new callable None **kwargs if type target is str getter lambda importer target else getter lambda target if not kwargs raise Value Error ' Mustsupplyatleastonekeywordargumentwithpatch multiple' items list kwargs items attribute new items[ 0 ]patcher patch getter attribute new spec create spec set autospec new callable {} patcher attribute name attributefor attribute new in items[ 1 ] this patcher patch getter attribute new spec create spec set autospec new callable {} this patcher attribute name attributepatcher additional patchers append this patcher return patcher",
   "question": "How do multiple patches perform ?",
   "answer": "in a single call"
  },
  {
   "id": 24867,
   "code": "@salt utils decorators which 'gzip' def gzip sourcefile template None runas None options None cmd ['gzip']if options cmd append options cmd append '{ 0 }' format sourcefile return salt ['cmd run'] cmd template template runas runas python shell False splitlines",
   "question": "What does the code use the gzip command ?",
   "answer": "to create gzip files template"
  },
  {
   "id": 17691,
   "code": "def Header Print MUA Details message mta None details []for header in MUA ID HEADERS value message get header if value value '' join [v for v in HP MUA ID SPLIT split value strip if not HP MUA ID IGNORE search v ] details extend [header value strip ] if not details if mta and mta[ 0 ] startswith ' Receivedbygoogle com' details extend [' Guessed' 'G Mail'] elif 'x-ms-tnef-correlator' in message or 'x-ms-has-attach' in message details extend [' Guessed' ' Exchange'] elif '@mailpile' in message get 'message-id' '' details extend [' Guessed' ' Mailpile'] return details",
   "question": "What does the code summarize ?",
   "answer": "what the message tells us directly about the mua"
  },
  {
   "id": 41943,
   "code": "def get server id servers identity for server in servers['items'] if identity in server['properties']['name'] server['id'] return server['id']return None",
   "question": "How do server uuid return if found ?",
   "answer": "by server name"
  },
  {
   "id": 45099,
   "code": "def make graph filename dep info sect gtype dependencies graph filename dep info sect append Paragraph '%simportsgraphhasbeenwrittento%s' % gtype filename",
   "question": "What does the code generate ?",
   "answer": "a dependencies graph"
  },
  {
   "id": 36500,
   "code": "def empty urlconf request t Template EMPTY URLCONF TEMPLATE name ' Empty URL Conftemplate' c Context {'project name' settings SETTINGS MODULE split ' ' [0 ]} return Http Response t render c mimetype 'text/html'",
   "question": "What does the code create ?",
   "answer": "an empty urlconf 404 error response"
  },
  {
   "id": 14392,
   "code": "def get Evaluated Expression Value value xml Element try return get Evaluated Expression Value By Split Line get Evaluator Split Words value xml Element except print ' Warning inget Evaluated Expression Valueinevaluatecouldnotgetavaluefor 'print valuetraceback print exc file sys stdout return None",
   "question": "What does the code evaluate ?",
   "answer": "the expression value"
  },
  {
   "id": 44866,
   "code": "def rule value def add attribute function if not hasattr function u'rule' function rule []function rule append value return functionreturn add attribute",
   "question": "What adds more rules ?",
   "answer": "this decorator"
  },
  {
   "id": 21252,
   "code": "def urljoin *segments url '/' if segments[ 0 ] startswith '/' else '' url + '/' join filter None force text s strip '/' for s in segments return url + '/' if settings APPEND SLASH else url",
   "question": "How does url segments join ?",
   "answer": "together"
  },
  {
   "id": 55237,
   "code": "def get g77 abi wrappers info wrapper sources []path os path abspath os path dirname file if needs g77 abi wrapper info wrapper sources + [os path join path 'src' 'wrap g77 abi f f' os path join path 'src' 'wrap g77 abi c c' ]if uses accelerate info wrapper sources + [os path join path 'src' 'wrap accelerate c c' os path join path 'src' 'wrap accelerate f f' ]elif uses mkl info wrapper sources + [os path join path 'src' 'wrap dummy accelerate f' ]else raise Not Implemented Error ' Donotknowhowtohandle LAPACK%sonmacosx' % info else wrapper sources + [os path join path 'src' 'wrap dummy g77 abi f' os path join path 'src' 'wrap dummy accelerate f' ]return wrapper sources",
   "question": "What is containing fortran abi wrapper routines ?",
   "answer": "source files"
  },
  {
   "id": 50352,
   "code": "def task enable root logins distribution commands [sudo from args ['sed' '-i' '1 i Permit Root Loginyes' '/etc/ssh/sshd config'] ]if is systemd distribution distribution commands append sudo from args ['systemctl' 'restart' 'sshd'] else commands append sudo from args ['service' 'ssh' 'restart'] return sequence commands",
   "question": "What do the ssh server configure ?",
   "answer": "to allow root login"
  },
  {
   "id": 50855,
   "code": "def add loss summaries total loss loss averages tf train Exponential Moving Average 0 9 name 'avg' losses tf get collection 'losses' loss averages op loss averages apply losses + [total loss] for l in losses + [total loss] tf scalar summary l op name + ' raw ' l tf scalar summary l op name loss averages average l return loss averages op",
   "question": "What does the code add ?",
   "answer": "summaries for losses in cifar-10 model"
  },
  {
   "id": 3665,
   "code": "def extract possible number number match VALID START CHAR PATTERN search number if match number number[match start ]trailing chars match UNWANTED END CHAR PATTERN search number if trailing chars match number number[ trailing chars match start ]second number match SECOND NUMBER START PATTERN search number if second number match number number[ second number match start ]return numberelse return U EMPTY STRING",
   "question": "In which direction did the string pass ?",
   "answer": "in"
  },
  {
   "id": 15010,
   "code": "def get service result parser run utils run global service result parsertry return service result parserexcept Name Error result parser result parsers[get name of init run ] service result parser Service Result Parser result parser return service result parser",
   "question": "How does the code get the serviceresultparser ?",
   "answer": "using the auto - detect init command"
  },
  {
   "id": 53857,
   "code": "def remove statements evaluator stmt name types set check instance Noneif isinstance stmt er Instance Element and stmt is class var check instance stmt instancestmt stmt varpep 0484 types pep 0484 find type from comment hint assign evaluator stmt name if pep 0484 types return pep 0484 typestypes evaluator eval statement stmt seek name name if check instance is not None types set [ er get instance el evaluator check instance a True if isinstance a er Function tree Function else a for a in types] return types",
   "question": "What are being stripped the part ?",
   "answer": "statements"
  },
  {
   "id": 2103,
   "code": "@register tagdef regroup parser token bits token split contents if len bits 6 raise Template Syntax Error \"'regroup'tagtakesfivearguments\" target parser compile filter bits[ 1 ] if bits[ 2 ] 'by' raise Template Syntax Error \"secondargumentto'regroup'tagmustbe'by'\" if bits[ 4 ] 'as' raise Template Syntax Error \"next-to-lastargumentto'regroup'tagmustbe'as'\" var name bits[ 5 ]expression parser compile filter var name + VARIABLE ATTRIBUTE SEPARATOR + bits[ 3 ] return Regroup Node target expression var name",
   "question": "How does the code regroup a list of alike objects ?",
   "answer": "by a common attribute"
  },
  {
   "id": 52759,
   "code": "def get powercfg minute values scheme guid subguid safe name if scheme is None scheme get current scheme if grains ['osrelease'] '7 ' cmd 'powercfg/q{ 0 }{ 1 }' format scheme guid else cmd 'powercfg/q{ 0 }{ 1 }{ 2 }' format scheme guid subguid out salt ['cmd run'] cmd python shell False split out split '\\r\\n\\r\\n' if len split > 1 for s in split if safe name in s or subguid in s out sbreakelse out split[ 0 ]raw settings re findall ' Power Setting Index [0 - 9 a-fx]+ ' out return {'ac' int raw settings[ 0 ] 0 / 60 'dc' int raw settings[ 1 ] 0 / 60 }",
   "question": "Where does the ac / dc values return ?",
   "answer": "in an array"
  },
  {
   "id": 20673,
   "code": "def demo sent subjectivity text from nltk classify import Naive Bayes Classifierfrom nltk tokenize import regexpword tokenizer regexp Whitespace Tokenizer try sentim analyzer load 'sa subjectivity pickle' except Lookup Error print ' Cannotfindthesentimentanalyzeryouwanttoload 'print ' Traininganewoneusing Naive Bayes Classifier 'sentim analyzer demo subjectivity Naive Bayes Classifier train True tokenized text [word lower for word in word tokenizer tokenize text ]print sentim analyzer classify tokenized text",
   "question": "What does the code classify as subjective or objective using a stored sentimentanalyzer ?",
   "answer": "a single sentence"
  },
  {
   "id": 26290,
   "code": "def depth first search set tasks current task visited visited add current task if current task in set tasks['still pending not ext'] upstream failure Falseupstream missing dependency Falseupstream run by other worker Falseupstream scheduling error Falsefor task in current task requires if task not in visited depth first search set tasks task visited if task in set tasks['ever failed'] or task in set tasks['upstream failure'] set tasks['upstream failure'] add current task upstream failure Trueif task in set tasks['still pending ext'] or task in set tasks['upstream missing dependency'] set tasks['upstream missing dependency'] add current task upstream missing dependency Trueif task in set tasks['run by other worker'] or task in set tasks['upstream run by other worker'] set tasks['upstream run by other worker'] add current task upstream run by other worker Trueif task in set tasks['scheduling error'] set tasks['upstream scheduling error'] add current task upstream scheduling error Trueif not upstream failure and not upstream missing dependency and not upstream run by other worker and not upstream scheduling error and current task not in set tasks['run by other worker'] set tasks['not run'] add current task",
   "question": "Till when are tasks pending why ?",
   "answer": "still"
  },
  {
   "id": 17546,
   "code": "def config shortcut action context name parent keystr get shortcut context name qsc Q Shortcut Q Key Sequence keystr parent action qsc set Context Qt Widget With Children Shortcut sc Shortcut data qsc context name return sc",
   "question": "Where will the data contained in this tuple be registered ?",
   "answer": "in our shortcuts preferences page"
  },
  {
   "id": 4337,
   "code": "def show instance name call None if call 'action' raise Salt Cloud System Exit ' Theshow instanceactionmustbecalledwith-aor--action ' nodes list nodes full if name not in nodes return {}if 'name' not in nodes[name] nodes[name]['name'] nodes[name]['id']try utils ['cloud cache node'] nodes[name] active provider name opts except Type Error log warning ' Unabletoshowcachenodedata thismaybebecausethenodehasbeendeleted' return nodes[name]",
   "question": "What is concerning an instance ?",
   "answer": "the provider"
  },
  {
   "id": 13826,
   "code": "def get pending file rename vnames ' Pending File Rename Operations' ' Pending File Rename Operations 2 ' key 'SYSTEM\\\\ Current Control Set\\\\ Control\\\\ Session Manager'for vname in vnames reg ret salt ['reg read value'] 'HKLM' key vname if reg ret['success'] log debug ' Foundkey %s' key if reg ret['vdata'] and reg ret['vdata'] ' valuenotset ' return Trueelse log debug ' Unabletoaccesskey %s' key return False",
   "question": "What require a reboot ?",
   "answer": "pending file rename operations"
  },
  {
   "id": 37170,
   "code": "def ping *args **kw return True",
   "question": "What does the code p ?",
   "answer": "the lamps"
  },
  {
   "id": 47049,
   "code": "def sensitive variables *variables def decorator func @functools wraps func def sensitive variables wrapper *func args **func kwargs if variables sensitive variables wrapper sensitive variables variableselse sensitive variables wrapper sensitive variables ' ALL 'return func *func args **func kwargs return sensitive variables wrapperreturn decorator",
   "question": "Where did which variables use ?",
   "answer": "in the decorated function"
  },
  {
   "id": 20622,
   "code": "def get file path dest saltenv 'base' makedirs False template None gzip None **kwargs path dest render filenames path dest saltenv template **kwargs path senv salt utils url split env path if senv saltenv senvif not hash file path saltenv return ''else return client get file path dest makedirs saltenv gzip",
   "question": "What did the code use ?",
   "answer": "to get a single file from the salt master"
  },
  {
   "id": 13934,
   "code": "def prepare rgba array arr arr np asanyarray arr if arr ndim not in [3 4] or arr shape[ -1 ] 4 msg 'theinputarraymusthaveashape [ ]4 got{ 0 }' format arr shape raise Value Error msg return dtype img as float arr",
   "question": "What does the code check to be rgba ?",
   "answer": "the shape of the array"
  },
  {
   "id": 42954,
   "code": "def run suite suite testclass None if verbose runner unittest Text Test Runner sys stdout verbosity 2 else runner Basic Test Runner result runner run suite if not result was Successful if len result errors 1 and not result failures err result errors[ 0 ][ 1 ]elif len result failures 1 and not result errors err result failures[ 0 ][ 1 ]else if testclass is None msg 'errorsoccurred runinverbosemodefordetails'else msg 'errorsoccurredin%s %s' % testclass module testclass name raise Test Failed msg raise Test Failed err",
   "question": "In which direction do tests run ?",
   "answer": "from a unittest"
  },
  {
   "id": 32717,
   "code": "@register inclusion tag 'admin/prepopulated fields js html' takes context True def prepopulated fields js context prepopulated fields []if context['add'] and 'adminform' in context prepopulated fields extend context['adminform'] prepopulated fields if 'inline admin formsets' in context for inline admin formset in context['inline admin formsets'] for inline admin form in inline admin formset if inline admin form original is None prepopulated fields extend inline admin form prepopulated fields context update {'prepopulated fields' prepopulated fields} return context",
   "question": "What should render javascript for the prepopulated fields for both the admin form and inlines ?",
   "answer": "a list"
  },
  {
   "id": 25928,
   "code": "def group membership s3 db hrm configure pr group membership table db pr group membershipgtable db pr grouphtable s3 db hrm human resources 3 filter gtable system False & gtable group type 3 & htable type 1 & htable person id table person id def prep r if r method in 'create' 'create popup' 'update' 'update popup' person id get vars get '~ person id' None if person id field table person idfield default person idfield readable field writable Falsereturn Trues 3 prep prepoutput s3 rest controller 'pr' 'group membership' csv stylesheet 'hrm' 'group membership xsl' csv template 'group membership' return output",
   "question": "What does membership controller use ?",
   "answer": "the group_membership table"
  },
  {
   "id": 42129,
   "code": "def Define d flavor if flavor 'win' d d replace '#' '\\\\% 03 o' % ord '#' return Quote Shell Argument ninja syntax escape '-D' + d flavor",
   "question": "What does the code take ?",
   "answer": "a preprocessor define"
  },
  {
   "id": 30071,
   "code": "def cql typename casstypename return lookup casstype casstypename cql parameterized type",
   "question": "What does the code translate into a cql - style type specifier ?",
   "answer": "a cassandra - style type specifier"
  },
  {
   "id": 9363,
   "code": "def normalize points for row in points row / points[ -1 ]return points",
   "question": "Where does the code normalize a collection of points so that last row = 1 ?",
   "answer": "in homogeneous coordinates"
  },
  {
   "id": 17359,
   "code": "def do Scp Create po global g created SC Pscp Scp Create get option po 'binding string' get option po 'service class' get option po 'account name sam' None keywords get option po 'keywords' None g created SCP scpreturn scp distinguished Name",
   "question": "What does the code create ?",
   "answer": "a service connection point"
  },
  {
   "id": 9150,
   "code": "def get Loop Layers archivable Objects import Radius layer Height maximum Z should Print Warning z zone Arrangement loop Layers []while z < maximum Z triangle mesh get Loop Layer Append loop Layers z loops get Empty Z Loops archivable Objects import Radius True z zone Arrangement z + layer Heightreturn loop Layers",
   "question": "What does the code get ?",
   "answer": "loop layers"
  },
  {
   "id": 16162,
   "code": "def vb get network adapters machine name None machine None if machine name machine vb get box find Machine machine name network adapters []for i in range vb get max network slots try inetwork adapter machine get Network Adapter i network adapter vb xpcom to attribute dict inetwork adapter 'I Network Adapter' network adapter['properties'] inetwork adapter get Properties '' network adapters append network adapter except Exception passreturn network adapters",
   "question": "For what purpose is a valid machine_name or a machine needed ?",
   "answer": "to make this work"
  },
  {
   "id": 48823,
   "code": "def append period text if text[ -1 ] '\"' return text[ 0 -1 ] + ' \"' return text",
   "question": "When does the code append a period ?",
   "answer": "at the end of the sentence"
  },
  {
   "id": 36789,
   "code": "def psql query query user None host None port None maintenance db None password None runas None ret []csv query 'COPY {0 } TOSTDOUTWITHCSVHEADER' format query strip rstrip ' ' cmdret psql prepare and run ['-v' 'datestyle ISO MDY' '-c' csv query] runas runas host host user user port port maintenance db maintenance db password password if cmdret['retcode'] > 0 return retcsv file String IO cmdret['stdout'] header {}for row in csv reader csv file delimiter ' ' quotechar '\"' if not row continueif not header header rowcontinueret append dict zip header row return ret",
   "question": "What does the code run ?",
   "answer": "an sql - query"
  },
  {
   "id": 35322,
   "code": "def start service service name os cmd ['/usr/bin/openstack-service' 'start' service name]return salt ['cmd retcode'] os cmd 0",
   "question": "When do openstack service start ?",
   "answer": "immediately"
  },
  {
   "id": 17219,
   "code": "def unique everseen iterable key None seen set seen add seen addif key is None for element in ifilterfalse seen contains iterable seen add element yield element else for element in iterable k key element if k not in seen seen add k yield element",
   "question": "What do unique_everseen - > iterator get ?",
   "answer": "unique elements"
  },
  {
   "id": 48956,
   "code": "def configured cluster for nodes reactor certificates nodes dataset backend dataset backend configuration dataset backend config file provider None logging config None default volume size get default volume size dataset backend configuration cluster Cluster all nodes pvector nodes control node nodes[ 0 ] agent nodes nodes dataset backend dataset backend default volume size default volume size certificates certificates dataset backend config file dataset backend config file configuring perform make dispatcher reactor configure cluster cluster dataset backend configuration provider logging config configuring add Callback lambda ignored cluster return configuring",
   "question": "Where do flocker services run ?",
   "answer": "on the right nodes"
  },
  {
   "id": 38311,
   "code": "def dead code elimination graph du ud for node in graph rpo for i ins in node get loc with ins [ ] reg ins get lhs if reg is not None if reg i not in du if ins is call ins remove defined var elif ins has side effect continueelse update chain graph i du ud graph remove ins i",
   "question": "What does the code run ?",
   "answer": "a dead code elimination pass"
  },
  {
   "id": 28077,
   "code": "def py 3 path path if isinstance path six text type return pathassert isinstance path bytes if six PY 2 return pathreturn os fsdecode path",
   "question": "What does the code convert to unicode on python 3 only ?",
   "answer": "a bytestring path"
  },
  {
   "id": 38242,
   "code": "def write global name obj cache path global cache path write cache path name obj",
   "question": "How does the code write the object to the global sublime cache path ?",
   "answer": "using pickle arguments"
  },
  {
   "id": 26175,
   "code": "@register tag 'static' def do static parser token return Static Files Node handle token parser token",
   "question": "How does a template tag return the url to a file ?",
   "answer": "using staticfiles storage backend usage"
  },
  {
   "id": 23723,
   "code": "def init Target Set schemata size test set []for in range size test list random randint 0 1 for in range len schemata for i x in enumerate schemata if x '0 ' test[i] 0elif x '1 ' test[i] 1test set append test return test set",
   "question": "How did a target set ?",
   "answer": "with noisy string"
  },
  {
   "id": 14373,
   "code": "def test predict tpot obj TPOT Classifier try tpot obj predict testing features assert Falseexcept Value Error pass",
   "question": "What raises a valueerror when no optimized pipeline exists ?",
   "answer": "the tpot predict function"
  },
  {
   "id": 9065,
   "code": "def is library missing name path module name rsplit ' ' 1 try package import module path return not module has submodule package module except Import Error return is library missing path",
   "question": "Can library that failed to load be found under any templatetags directory ?",
   "answer": "No"
  },
  {
   "id": 33392,
   "code": "def sem a axis 0 ddof 1 a axis chk asarray a axis n a count axis axis s a std axis axis ddof ddof / ma sqrt n return s",
   "question": "What does the code calculate ?",
   "answer": "the standard error of the mean of the input array"
  },
  {
   "id": 32343,
   "code": "def remove grunt files for filename in [' Gruntfile js'] os remove os path join PROJECT DIRECTORY filename",
   "question": "For what purpose did files need ?",
   "answer": "for grunt"
  },
  {
   "id": 5957,
   "code": "def pytest generate tests metafunc fsz [3 4 7]dil [2 3]strides [1 2 3 6]if 'fargs tests' in metafunc fixturenames fargs itt product fsz dil strides metafunc parametrize 'fargs tests' fargs",
   "question": "What does the code build ?",
   "answer": "a list of test arguments"
  },
  {
   "id": 49530,
   "code": "def collection rights to search dict rights doc {}if rights status rights manager ACTIVITY STATUS PUBLICIZED doc['is'] 'featured'return doc",
   "question": "How does a search dict return ?",
   "answer": "with information about the collection rights"
  },
  {
   "id": 3087,
   "code": "def prefetch one level instances prefetcher attname rel qs rel obj attr instance attr single cache name prefetcher get prefetch queryset instances additional prl getattr rel qs ' prefetch related lookups' [] if additional prl rel qs prefetch related lookups []all related objects list rel qs rel obj cache {}for rel obj in all related objects rel attr val rel obj attr rel obj rel obj cache setdefault rel attr val [] append rel obj for obj in instances instance attr val instance attr obj vals rel obj cache get instance attr val [] if single setattr obj cache name vals[ 0 ] if vals else None else qs getattr obj attname all qs result cache valsqs prefetch done Trueobj prefetched objects cache[cache name] qsreturn all related objects additional prl",
   "question": "What runs prefetches on all instances ?",
   "answer": "helper function for prefetch_related_objects"
  },
  {
   "id": 7185,
   "code": "def Date From Ticks ticks return Date *time localtime ticks [ 3]",
   "question": "What do an object hold ?",
   "answer": "a date value from the given ticks value"
  },
  {
   "id": 10016,
   "code": "def make Fake K Queue test K Queue test K Event @implementer IK Queue class Fake K Queue object kqueue test K Queuekevent test K Eventreturn Fake K Queue",
   "question": "What does the code create ?",
   "answer": "a fake that implements l{_ikqueue }"
  },
  {
   "id": 41120,
   "code": "def get disconnect url provider id association id backend name get enabled provider provider id backend nameif association id return get url 'social disconnect individual' backend name url params {'association id' association id} else return get url 'social disconnect' backend name",
   "question": "What does the endpoint start ?",
   "answer": "the disconnect pipeline"
  },
  {
   "id": 10484,
   "code": "def run script with context script path cwd context extension os path splitext script path contents io open script path 'r' encoding 'utf- 8 ' read with tempfile Named Temporary File delete False mode 'wb' suffix extension as temp env Strict Environment context context keep trailing newline True template env from string contents output template render **context temp write output encode 'utf- 8 ' run script temp name cwd",
   "question": "When does the code execute a script ?",
   "answer": "after rendering it with jinja"
  },
  {
   "id": 51909,
   "code": "def get members mailchimp list id status mc get members mailchimp list Membersmembers set for page in itertools count response mc get members id list id status status start page limit BATCH SIZE data response get 'data' [] if not data breakmembers update d['email'] for d in data return members",
   "question": "For what purpose did a mailchimp list i d and a user status give the code ?",
   "answer": "to filter on"
  },
  {
   "id": 44596,
   "code": "def missing whitespace after import keyword logical line line logical lineindicator 'import 'if line startswith 'from' found line find indicator if -1 < found pos found + len indicator - 1 yield pos 'E 275 missingwhitespaceafterkeyword'",
   "question": "What should multiple imports in form from x import have ?",
   "answer": "space between import statement and parenthesised name list"
  },
  {
   "id": 27666,
   "code": "def check valid naming pattern None multi None anime type None if pattern is None pattern sickrage sr Core sr Config NAMING PATTER Nif anime type is None anime type sickrage sr Core sr Config NAMING ANIM Esickrage sr Core sr Logger debug u' Checkingwhetherthepattern' + pattern + u'isvalid' return validate name pattern multi anime type",
   "question": "In which direction can the name be parsed to its original form for both single and multi episodes ?",
   "answer": "back"
  },
  {
   "id": 39086,
   "code": "def check cli module cli vlanid module params['pn vlanid']show cli + 'vlan-showid%sformatid scopeno-show-headers' % str vlanid show shlex split show out module run command show [1 ]out out split global VLAN EXIST Sif str vlanid in out VLAN EXISTS Trueelse VLAN EXISTS False",
   "question": "How does this method check for idempotency ?",
   "answer": "using the vlan - show command"
  },
  {
   "id": 4163,
   "code": "def image member find context image id None member None status None include deleted False session get session members image member find context session image id member status include deleted return [ image member format m for m in members]",
   "question": "What meet the given criteria ?",
   "answer": "all members"
  },
  {
   "id": 25216,
   "code": "def import helper mod name try sys stdout STDOUT FAKE import mod name finally sys stdout STDOUT BAK",
   "question": "What used to temporarily override stdout before importing a module ?",
   "answer": "helper function"
  },
  {
   "id": 25009,
   "code": "def download and uncompress tarball tarball url dataset dir filename tarball url split '/' [ -1 ]filepath os path join dataset dir filename def progress count block size total size sys stdout write '\\r>> Downloading%s% 1f%%' % filename float count * block size / float total size * 100 0 sys stdout flush filepath urllib request urlretrieve tarball url filepath progress print statinfo os stat filepath print ' Successfullydownloaded' filename statinfo st size 'bytes ' tarfile open filepath 'r gz' extractall dataset dir",
   "question": "What does the code download ?",
   "answer": "the tarball_url"
  },
  {
   "id": 39052,
   "code": "def setup hass config global DAT Aconf config[DOMAIN]host conf get CONF HOST port conf get CONF PORT DATA APCUP Sd Data host port try DATA update no throttle True except Exception LOGGER exception ' Failurewhiletesting APCUP Sdstatusretrieval ' return Falsereturn True",
   "question": "What is enabling status retrieval ?",
   "answer": "a function"
  },
  {
   "id": 30674,
   "code": "def load pandas from pandas import Data Framefrom statsmodels tools import categoricaldata get data raw data categorical data col 'firm' drop True ds du process recarray pandas data endog idx 0 ds raw data Data Frame raw data return ds",
   "question": "What does the code load ?",
   "answer": "the grunfeld data"
  },
  {
   "id": 34676,
   "code": "def write flv header stream stream write 'FLV\\x 01 ' stream write '\\x 05 ' stream write '\\x 00 \\x 00 \\x 00 DCTB ' stream write '\\x 00 \\x 00 \\x 00 \\x 00 '",
   "question": "What does the code write ?",
   "answer": "the flv header to stream"
  },
  {
   "id": 16476,
   "code": "def color to reportlab color if isinstance color colors Color return colorelif isinstance color str if color startswith '0 x' color replace '0 x' '#' if len color 7 return colors Hex Color color else try return colors Hex Color color has Alpha True except Type Error raise Runtime Error ' Yourreportlabseemstobetooold try 2 7onwards' elif isinstance color tuple return colors Color *color return color",
   "question": "How does the passed color return ?",
   "answer": "in reportlab color format"
  },
  {
   "id": 36256,
   "code": "def colorize color text if config['color'] return colorize color text else return text",
   "question": "What does the code colorize if colored output is enabled ?",
   "answer": "text"
  },
  {
   "id": 752,
   "code": "def tax subtract price tax percentage 21 if price is None return Noneresult price / 100 + tax percentage / D 100 return result quantize D '0 01 ' ROUND HALF UP",
   "question": "What does the code subtract from the given price ?",
   "answer": "the given tax_percentage"
  },
  {
   "id": 44583,
   "code": "def IE Dispatcher s if len s < 1 return Raw s ietype ord s[ 0 ] cls ietypecls get ietype Raw if cls Raw and ietype & 128 128 cls IE Not Implemented TL Vreturn cls s",
   "question": "What does the code choose ?",
   "answer": "the correct information element class"
  },
  {
   "id": 47233,
   "code": "def format dot dotfilename format None cmd u\"dot-T%s-O'%s'\" % format dotfilename try Command Line cmd run except IO Error as ioe if u'couldnotbefound' in str ioe raise IO Error u\" Cannotdrawdirectedgraph executable'dot'isunavailable\" else raise ioeelse logger info u' Convertingdotfile %sto%sformat' % dotfilename format",
   "question": "What does the code dump ?",
   "answer": "a directed graph"
  },
  {
   "id": 7507,
   "code": "def test parent parent Parent t usertypes Timer parent assert t parent is parent",
   "question": "How is the parent set ?",
   "answer": "correctly"
  },
  {
   "id": 44676,
   "code": "def lookup session name label check rescue False if check rescue result lookup session name label + '-rescue' False if result return resultvm refs session call xenapi 'VM get by name label' name label n len vm refs if n 0 return Noneelif n > 1 raise exception Instance Exists name name label else return vm refs[ 0 ]",
   "question": "What does the code look up ?",
   "answer": "the instance"
  },
  {
   "id": 35190,
   "code": "def check call *args **kwargs retcode call *args **kwargs if retcode cmd kwargs get 'args' if cmd is None cmd args[ 0 ]raise Called Process Error retcode cmd",
   "question": "What does the code call with an optional timeout ?",
   "answer": "a program"
  },
  {
   "id": 49209,
   "code": "def redirect to login next login url LOGIN URL return Http Response Redirect '%s?%s %s' % login url REDIRECT FIELD NAME next",
   "question": "What does the code redirect to the login page ?",
   "answer": "the user"
  },
  {
   "id": 17486,
   "code": "def process fastq single end read file no barcode fastq read f sample id store unassigned False max bad run length 0 phred quality threshold 2 min per read length fraction 0 75 rev comp False seq max N 0 start seq id 0 filter bad illumina qual digit False log f None histogram f None phred offset None fake barcodes cycle ['@' 'AAAAAAAAAAAA' '+' 'CCCCCCCCCCCC'] barcode to sample id {'AAAAAAAAAAAA' sample id}for e in process fastq single end read file fastq read f fake barcodes barcode to sample id store unassigned store unassigned max bad run length max bad run length phred quality threshold phred quality threshold min per read length fraction min per read length fraction rev comp rev comp rev comp barcode False seq max N seq max N start seq id start seq id filter bad illumina qual digit filter bad illumina qual digit log f log f histogram f histogram f barcode correction fn None max barcode errors 0 strict header match False phred offset phred offset yield e",
   "question": "What simulates a barcode file to allow us to re - use the quality filtering code when not demultiplexing when a single sample has been run in a lane ?",
   "answer": "this code"
  },
  {
   "id": 50370,
   "code": "def generate jwt credentials App Assertion Credentials 'https //www googleapis com/auth/iam' http auth credentials authorize httplib 2 Http service build service Name 'iam' version 'v 1 ' http http auth now int time time header json json dumps {'typ' 'JWT' 'alg' 'RS 256 '} payload json json dumps {'iat' now 'exp' now + 3600 'iss' SERVICE ACCOUNT EMAIL 'sub' SERVICE ACCOUNT EMAIL 'aud' 'echo endpoints sample google com' 'email' SERVICE ACCOUNT EMAIL} header And Payload '{} {}' format base 64 urlsafe b64 encode header json base 64 urlsafe b64 encode payload json slist service projects service Accounts sign Blob name SERVICE ACCOUNT body {'bytes To Sign' base 64 b64 encode header And Payload } res slist execute signature base 64 urlsafe b64 encode base 64 decodestring res['signature'] signed jwt '{} {}' format header And Payload signature return signed jwt",
   "question": "What does the code generate ?",
   "answer": "a signed json web token using a service account"
  },
  {
   "id": 42006,
   "code": "def atomic group remove labels id labels label objs models Label smart get bulk labels models Atomic Group smart get id label set remove *label objs",
   "question": "What does the code remove from atomic group ?",
   "answer": "labels"
  },
  {
   "id": 22293,
   "code": "def set response cookie path None path header None name 'session id' timeout 60 domain None secure False httponly False cookie cherrypy serving response cookiecookie[name] cherrypy serving session idcookie[name]['path'] path or cherrypy serving request headers get path header or '/' if False and timeout e time time + timeout * 60 cookie[name]['expires'] httputil HTTP Date e if domain is not None cookie[name]['domain'] domainif secure cookie[name]['secure'] 1if httponly if not cookie[name] is Reserved Key 'httponly' raise Value Error ' Thehttponlycookietokenisnotsupported ' cookie[name]['httponly'] 1",
   "question": "What did the code set for the client ?",
   "answer": "a response cookie"
  },
  {
   "id": 7310,
   "code": "def extract dependencies reg assignments return sorted [k for k v in assignments items if v reg ]",
   "question": "How do all registers depend on the specified register ?",
   "answer": "directly"
  },
  {
   "id": 19234,
   "code": "def cache on disk t cache disk 'time' lambda time ctime time expire 5 return dict time t link A 'clicktoreload' href URL r request",
   "question": "What caches on disk ?",
   "answer": "the output of the lambda function"
  },
  {
   "id": 28492,
   "code": "def delete bookmark user usage key bookmark Bookmark objects get user user usage key usage key bookmark delete track event 'edx bookmark removed' bookmark",
   "question": "What does the code delete ?",
   "answer": "a bookmark"
  },
  {
   "id": 10365,
   "code": "def test live monitoring verify zmq p mp Process target train mlp p start correct result set ['train objective' 'train y col norms max' 'train y row norms min' 'train y nll' 'train y col norms mean' 'train y max max class' 'train y min max class' 'train y row norms max' 'train y misclass' 'train y col norms min' 'train y row norms mean' 'train y mean max class' 'learning rate' 'training seconds this epoch' 'total seconds last epoch'] monitor lm Live Monitor result set monitor list channels data if result correct result raise Value Error str result assert result correct result monitor lm Live Monitor monitor update channels ['train objective'] start 0 end 2 assert len monitor channels['train objective'] val record 2 monitor lm Live Monitor monitor update channels ['train objective'] start 1 end 2 assert len monitor channels['train objective'] val record 1 p join assert raises Assertion Error monitor update channels 0 assert raises Assertion Error monitor update channels [] assert raises Assertion Error monitor update channels ['train objective'] start 2 end 1",
   "question": "What starts a secondary process to train an mlp with the livemonitoring train extension ?",
   "answer": "function"
  },
  {
   "id": 29735,
   "code": "@nottestdef normalize test names test suite for test in test suite tests test methodname re sub '0 x *? >' '0 xfffffff>' test methodname",
   "question": "What do tests which are generated on the fly have ?",
   "answer": "names like : foo"
  },
  {
   "id": 212,
   "code": "def render template multiple times request return Http Response render to string 'base html' + render to string 'base html'",
   "question": "What does a view render multiple times ?",
   "answer": "a template"
  },
  {
   "id": 47323,
   "code": "def seconds to str value if value < 60 return '%sseconds' % value elif value < 3600 return '%sminutes' % value / 60 else return '%shoursand%sminutes' % value / 3600 value % 3600 / 60",
   "question": "What is describing the amount of time ?",
   "answer": "a simple simple string"
  },
  {
   "id": 47164,
   "code": "def detection limit index obs cohn if cohn shape[ 0 ] > 0 index numpy where cohn['lower dl'] < obs det limit index index[ -1 ]else det limit index 0return det limit index",
   "question": "What does the code locate ?",
   "answer": "the corresponding detection limit for each observation"
  },
  {
   "id": 17261,
   "code": "def add sld boilerplate symbolizer return '\\n< Styled Layer Descriptorversion \"1 0 0\"xmlns \"http //www opengis net/sld\"xmlns ogc \"http //www opengis net/ogc\"\\nxmlns xlink \"http //www w3 org/ 1999 /xlink\"xmlns xsi \"http //www w3 org/ 2001 /XML Schema-instance\"\\nxsi schema Location \"http //www opengis net/sldhttp //schemas opengis net/sld/ 1 0 0/ Styled Layer Descriptor xsd\">\\n< Named Layer>\\n< Name>% name s</ Name>\\n< User Style>\\n< Name>% name s</ Name>\\n< Title>% name s</ Title>\\n< Feature Type Style>\\n< Rule>\\n' + symbolizer + '\\n</ Rule>\\n</ Feature Type Style>\\n</ User Style>\\n</ Named Layer>\\n</ Styled Layer Descriptor>\\n'",
   "question": "What do an xml snippet represent in the appropriate elements ?",
   "answer": "a single symbolizer"
  },
  {
   "id": 32806,
   "code": "def distributed server only handler def distributed server only wrapper fn *args **kwargs if settings CENTRAL SERVER raise Http 404 ' Thispathisonlyavailableondistributedservers ' return handler *args **kwargs return distributed server only wrapper fn",
   "question": "What marks a function for use only on a distributed server ?",
   "answer": "decorator"
  },
  {
   "id": 49935,
   "code": "@pytest mark django dbdef test data tp checks english tp Translation Project Factory project Project DB Factory source language english language Language DB Factory check data TP Checks Data objects create tp tp assert repr check data '<TP Checks Data %s>' % tp pootle path",
   "question": "Do you add a duplicate file extension ?",
   "answer": "No"
  },
  {
   "id": 11708,
   "code": "def date range start end if getattr start 'date' None is not None start start date if getattr end 'date' None is not None end end date days end - start days + 1 return start + datetime timedelta days d for d in xrange days",
   "question": "What do an iterator provide ?",
   "answer": "the dates between start and end"
  },
  {
   "id": 33271,
   "code": "def add Faces By Concave Loop faces indexed Loop if len indexed Loop < 3 returnremaining Loop indexed Loop[ ]while len remaining Loop > 2 remaining Loop get Remaining Loop Add Face faces remaining Loop",
   "question": "What does the code add from a polygon which is concave ?",
   "answer": "faces"
  },
  {
   "id": 6973,
   "code": "def shutdown handler List handler List for wr in reversed handler List[ ] try h wr if h try h acquire h flush h close except OS Error Value Error passfinally h release except if raise Exceptions raise",
   "question": "What does the code perform ?",
   "answer": "any cleanup actions in the logging system"
  },
  {
   "id": 40979,
   "code": "def portable Install reactor Portable Gtk Reactor from twisted internet main import install Reactorinstall Reactor reactor return reactor",
   "question": "What do the twisted mainloop configure ?",
   "answer": "to be run inside the gtk mainloop"
  },
  {
   "id": 19558,
   "code": "def Add Action Step actions dict inputs outputs description command assert inputsaction {'inputs' inputs 'outputs' outputs 'description' description 'command' command}chosen input inputs[ 0 ]if chosen input not in actions dict actions dict[chosen input] []actions dict[chosen input] append action",
   "question": "What merges into an existing list of actions ?",
   "answer": "action"
  },
  {
   "id": 55159,
   "code": "def get Tree Builder tree Type implementation None **kwargs tree Type tree Type lower if tree Type not in tree Builder Cache if tree Type 'dom' import domif implementation None from xml dom import minidomimplementation minidomreturn dom get Dom Module implementation **kwargs Tree Builderelif tree Type 'simpletree' import simpletreetree Builder Cache[tree Type] simpletree Tree Builderelif tree Type 'beautifulsoup' import souptree Builder Cache[tree Type] soup Tree Builderelif tree Type 'lxml' import etree lxmltree Builder Cache[tree Type] etree lxml Tree Builderelif tree Type 'etree' if implementation None try import xml etree c Element Tree as E Texcept Import Error try import xml etree Element Tree as E Texcept Import Error try import c Element Tree as E Texcept Import Error import elementtree Element Tree as E Timplementation E Timport etreereturn etree get E Tree Module implementation **kwargs Tree Builderelse raise Value Error ' Unrecognisedtreebuilder\"%s\"' % tree Type return tree Builder Cache get tree Type",
   "question": "What does the code get ?",
   "answer": "a treebuilder class for various types of tree with built - in support treetype - the name of the tree type required"
  },
  {
   "id": 41874,
   "code": "def split args args args dict {}for arg in args split arg arg split ' ' 1 if len split arg > 1 value split arg[ 1 ]else value Trueargs dict[split arg[ 0 ]] valuereturn args dict",
   "question": "What is an argument name a dictionary ?",
   "answer": "each key"
  },
  {
   "id": 33270,
   "code": "def make temp fname fname None suffix os path split fname [ -1 ] fd temp prefix tempfile mkstemp prefix 'tmp' suffix suffix return temp prefix",
   "question": "What interpret it ?",
   "answer": "tools"
  },
  {
   "id": 39909,
   "code": "def issues closed since period timedelta days 365 project 'statsmodels/statsmodels' pulls False which 'pulls' if pulls else 'issues' if isinstance period timedelta since round hour datetime utcnow - period else since periodurl 'https //api github com/repos/%s/%s?state closed&sort updated&since %s&per page %i' % project which since strftime ISO 8601 PER PAGE allclosed get paged request url headers make auth header filtered [i for i in allclosed if parse datetime i['closed at'] > since ]if pulls filtered [i for i in filtered if parse datetime i['merged at'] > since ]filtered [i for i in filtered if i['base']['ref'] 'master' ]else filtered [i for i in filtered if not is pull request i ]return filtered",
   "question": "What does the code get since a particular point in time ?",
   "answer": "all issues closed"
  },
  {
   "id": 13035,
   "code": "def full rary tree r n create using None G nx empty graph n create using G add edges from tree edges n r return G",
   "question": "What does the code create ?",
   "answer": "a full r - ary tree of n vertices"
  },
  {
   "id": 24504,
   "code": "def test scharr vertical i j np mgrid[ -5 6 -5 6]image j > 0 astype float result filters scharr image * np sqrt 2 j[ np abs i 5 ] 10000 assert allclose result[ j 0 ] 1 assert np all result[ np abs j > 1 ] 0",
   "question": "Where be scharr be a vertical line ?",
   "answer": "on a vertical edge"
  },
  {
   "id": 49971,
   "code": "def cast to server context server params topic msg return get impl cast to server CONF context server params topic msg",
   "question": "Does a remote method return anything ?",
   "answer": "No"
  },
  {
   "id": 10587,
   "code": "def get words from dictionary lemmas words set for lemma in lemmas words update set lemmas[lemma] return words",
   "question": "What does the code get ?",
   "answer": "original set of words used for analysis"
  },
  {
   "id": 35348,
   "code": "def memoize traffic **memoize kwargs def memoize traffic decorator fn def memoize traffic wrapper cls *args **kwargs method ' ' join cls name fn name actual memoize decorator memoize method **memoize kwargs actual memoize wrapper actual memoize decorator fn return actual memoize wrapper cls *args **kwargs return memoize traffic wrapperreturn memoize traffic decorator",
   "question": "What does the code determine automatically ?",
   "answer": "memoize key"
  },
  {
   "id": 51028,
   "code": "def find value ret dict key path None if path is None path keyelse path '{ 0 } {1 }' format path key ret []for ikey val in six iteritems ret dict if ikey key ret append {path val} if isinstance val list for item in val if isinstance item dict ret ret + find value item key path if isinstance val dict ret ret + find value val key path return ret",
   "question": "What finds key ?",
   "answer": "private method"
  },
  {
   "id": 34757,
   "code": "def create event service key None description None details None incident key None profile None trigger url 'https //events pagerduty com/generic/ 2010 - 04 - 15 /create event json'if isinstance details string types details yaml safe load details if isinstance details string types details {'details' details}ret json loads salt utils pagerduty query method 'POST' profile dict salt ['config option'] profile api key service key data {'service key' service key 'incident key' incident key 'event type' 'trigger' 'description' description 'details' details} url trigger url opts opts return ret",
   "question": "What does the code create ?",
   "answer": "an event in pagerduty"
  },
  {
   "id": 37,
   "code": "def replication group absent name wait 600 region None key None keyid None profile None **args ret {'name' name 'result' True 'comment' '' 'changes' {}}args dict [ k v for k v in args items if not k startswith ' ' ] exists salt ['boto 3 elasticache replication group exists'] name region region key key keyid keyid profile profile if exists if opts ['test'] ret['comment'] ' Replicationgroup{ 0 }wouldberemoved ' format name ret['result'] Nonereturn retdeleted salt ['boto 3 elasticache delete replication group'] name wait wait region region key key keyid keyid profile profile **args if deleted ret['changes']['old'] nameret['changes']['new'] Noneelse ret['result'] Falseret['comment'] ' Failedtodelete{ 0 }replicationgroup ' format name else ret['comment'] ' Replicationgroup{ 0 }alreadyabsent ' format name return ret",
   "question": "What does the code ensure ?",
   "answer": "a given cache cluster is deleted"
  },
  {
   "id": 39252,
   "code": "def skip if windows reason def decorator func return unittest skip If platform system not in [' Darwin' ' Linux'] reason func return decorator",
   "question": "What do decorator skip ?",
   "answer": "tests that should not be run on windows"
  },
  {
   "id": 21243,
   "code": "@memoizedef from theme name fallback None if hasattr Qt Gui Q Icon u'from Theme' base ext os path splitext name if fallback qicon Qt Gui Q Icon from Theme base icon fallback else qicon Qt Gui Q Icon from Theme base if not qicon is Null return qiconreturn icon fallback or name",
   "question": "How do an icon grab from the current theme ?",
   "answer": "with a fallback support older versions of qt checking for fromthemes availability"
  },
  {
   "id": 32021,
   "code": "def init Logger obj if inspect isclass obj my Class objelse my Class obj class logger logging get Logger ' ' join ['com numenta' my Class module my Class name ] return logger",
   "question": "What creates for the current object ?",
   "answer": "a logger object"
  },
  {
   "id": 1276,
   "code": "def create login url dest url None auth domain None federated identity None req user service pb Create Login URL Request resp user service pb Create Login URL Response if dest url req set destination url dest url else req set destination url '' if auth domain req set auth domain auth domain if federated identity req set federated identity federated identity try apiproxy stub map Make Sync Call 'user' ' Create Login URL' req resp except apiproxy errors Application Error as e if e application error user service pb User Service Error REDIRECT URL TOO LONG raise Redirect Too Long Errorelif e application error user service pb User Service Error NOT ALLOWED raise Not Allowed Errorelse raise ereturn resp login url",
   "question": "What does the code compute ?",
   "answer": "the login url for redirection"
  },
  {
   "id": 7331,
   "code": "def test notation regexp pattern re compile SCIENTIFIC NOTATION REGEXP matches [' 1 e 3 ' '1 E4 ' '- 1 e- 3 ' '2 3e+ 4 ' ' 2e 4 ']fails ['string' '4 ' '2 1' '1 2e 3 2' ' e4 ' 'a 3 e 4 ']for match in matches assert pattern match match for fail in fails assert not pattern match fail",
   "question": "How are by yaml not parsed scientific notations as floats ?",
   "answer": "correctly"
  },
  {
   "id": 23788,
   "code": "def convert nn val if val < 20 return to 19 [val]for dcap dval in k 20 + 10 * v for v k in enumerate tens if dval + 10 > val if val % 10 return dcap + '-' + to 19 [ val % 10 ] return dcap",
   "question": "What does the code convert a value < 100 ?",
   "answer": "to english"
  },
  {
   "id": 54504,
   "code": "def change root new root pathname if os name 'posix' if not os path isabs pathname return os path join new root pathname else return os path join new root pathname[ 1 ] elif os name 'nt' drive path os path splitdrive pathname if path[ 0 ] '\\\\' path path[ 1 ]return os path join new root path else raise Distutils Platform Error \"nothingknownaboutplatform'%s'\" % os name",
   "question": "How do pathname return ?",
   "answer": "with new_root prepended"
  },
  {
   "id": 52872,
   "code": "def install as gi import sysif 'gi repository' in const PREFIX returnfor mod in iterkeys sys modules if mod 'gi' or mod startswith 'gi ' raise Assertion Error 'pgihastobeimportedbeforegi' import pgiimport pgi repositorysys modules['gi'] pgisys modules['gi repository'] pgi repositoryconst PREFIX append 'gi repository'",
   "question": "What do the first gi import redirect gi imports ?",
   "answer": "to pgi"
  },
  {
   "id": 20153,
   "code": "@contextlib contextmanagerdef remove path on error path try yield except Exception with excutils save and reraise exception delete if exists path",
   "question": "How does that operate ?",
   "answer": "on path"
  },
  {
   "id": 15692,
   "code": "def add X Intersection Indexes From Loop Lists Y loop Lists x Intersection Index List y for loop List Index in xrange len loop Lists loop List loop Lists[loop List Index]add X Intersection Indexes From Loops Y loop List loop List Index x Intersection Index List y",
   "question": "What does the code add ?",
   "answer": "the x intersection indexes for the loop lists"
  },
  {
   "id": 3796,
   "code": "def add Value To Output depth key Input output value depth Start '' * depth output write '%s%s ' % depth Start key Input if value class dict output write '\\n' keys value keys keys sort for key in keys add Value To Output depth + 1 key output value[key] returnif value class list output write '\\n' for element Index element in enumerate value add Value To Output depth + 1 element Index output element returnoutput write '%s\\n' % value",
   "question": "What does the code add to the output ?",
   "answer": "value"
  },
  {
   "id": 15785,
   "code": "def test classify with open 'gsn sup example pkl' as f gsn pickle load f gsn Joint GSN convert gsn gsn corrupt switch Falseds MNIST which set 'test' mb data ds Xy ds yfor i in xrange 1 10 y hat gsn classify mb data trials i errors np abs y hat - y sum / 2 0 print i errors errors / mb data shape[ 0 ]",
   "question": "What performs at classification ?",
   "answer": "a gsn"
  },
  {
   "id": 37182,
   "code": "def subscription check subscriptions Subscription objects all for subscription in subscriptions subscription check status",
   "question": "How do assets depreciate ?",
   "answer": "automatically"
  },
  {
   "id": 5161,
   "code": "def Configure Remote Api From Server server path app id services None default auth domain None use remote datastore True if services is None services set ALL SERVICES else services set services unsupported services difference ALL SERVICES if unsupported raise Configuration Error ' Unsupportedservice s %s' % ' ' join unsupported os environ['APPLICATION ID'] app idos environ setdefault 'AUTH DOMAIN' default auth domain or 'gmail com' apiproxy stub map apiproxy apiproxy stub map API Proxy Stub Map if 'datastore v3 ' in services and use remote datastore services remove 'datastore v3 ' datastore stub Remote Datastore Stub server path apiproxy stub map apiproxy Register Stub 'datastore v3 ' datastore stub stub Remote Stub server path for service in services apiproxy stub map apiproxy Register Stub service stub",
   "question": "What does the code do to allow easy remote access to app engine apis ?",
   "answer": "necessary setup"
  },
  {
   "id": 5613,
   "code": "def test suggested multiple column names completer complete event text u'SELEC Tid fromcustom products'position len u'SELEC Tid ' result set completer get completions Document text text cursor position position complete event assert set result set testdata columns u'products' u'custom' + testdata functions + list testdata builtin functions + testdata keywords",
   "question": "In which direction do multiple columns select when ?",
   "answer": "from table"
  },
  {
   "id": 47966,
   "code": "def rref solve matlist variable constant K new matlist copy deepcopy matlist augmented augment new matlist constant K solution rref augmented K return col solution -1",
   "question": "What is using reduced row echelon form given a matrix of coefficients ?",
   "answer": "equations"
  },
  {
   "id": 31849,
   "code": "def tokenize wrapper input skip token NEWLINE token INDENT token DEDENT tokens tokenize generate tokens driver generate lines input next for quintuple in tokens type value start end line text quintupleif type not in skip yield quintuple",
   "question": "What is suppressing significant whitespace ?",
   "answer": "a string"
  },
  {
   "id": 53478,
   "code": "def redirect back url source domain parse data urlparse url domain parse data netlocquery parse data queryif source domain in domain or domain in source domain return urlquery item parse qs query if query item get 'url' return query item['url'][ 0 ]return url",
   "question": "How did news args direct to their site ?",
   "answer": "with the real news url"
  },
  {
   "id": 4535,
   "code": "def get Tree Walker tree Type implementation None **kwargs tree Type tree Type lower if tree Type not in tree Walker Cache if tree Type in 'dom' 'pulldom' 'simpletree' mod import tree Type globals tree Walker Cache[tree Type] mod Tree Walkerelif tree Type 'genshi' import genshistreamtree Walker Cache[tree Type] genshistream Tree Walkerelif tree Type 'beautifulsoup' import souptree Walker Cache[tree Type] soup Tree Walkerelif tree Type 'lxml' import lxmletreetree Walker Cache[tree Type] lxmletree Tree Walkerelif tree Type 'etree' import etreereturn etree get E Tree Module implementation **kwargs Tree Walkerreturn tree Walker Cache get tree Type",
   "question": "What does the code get ?",
   "answer": "a treewalker class for various types of tree with built - in support treetype - the name of the tree type required"
  },
  {
   "id": 49846,
   "code": "def main env args db with sentry env upsert env args db",
   "question": "How does the code load it into postgres ?",
   "answer": "using an ingenious algorithm"
  },
  {
   "id": 34911,
   "code": "def check binary probabilistic predictions y true y prob check consistent length y true y prob labels np unique y true if len labels > 2 raise Value Error ' Onlybinaryclassificationissupported Providedlabels%s ' % labels if y prob max > 1 raise Value Error 'y probcontainsvaluesgreaterthan 1 ' if y prob min < 0 raise Value Error 'y probcontainsvalueslessthan 0 ' return label binarize y true labels [ 0]",
   "question": "What does y_prob contain ?",
   "answer": "valid probabilities"
  },
  {
   "id": 50189,
   "code": "def setup config test control address u' 10 0 0 1' control port 1234 name None log config None if name is None name random name test ca set get credential sets [0 ]scratch directory test make temporary directory contents {u'control-service' {u'hostname' control address u'port' control port} u'dataset' {u'backend' u'zfs' u'name' name u'mount root' scratch directory child 'mount root' path u'volume config path' scratch directory child 'volume config json' path} u'version' 1}if log config is not None contents[u'logging'] log configtest config scratch directory child 'dataset-config yml' test config set Content yaml safe dump contents ca set copy to scratch directory node True test ca set ca set",
   "question": "What does the code create in a temporary directory ?",
   "answer": "a configuration file and certificates for a dataset agent"
  },
  {
   "id": 8241,
   "code": "def time 2 netscape t None if t is None t time time year mon mday hour min sec wday time gmtime t [ 7]return '%s% 02 d-%s-% 04 d% 02 d %02 d %02 d GMT' % DAYS[wday] mday MONTHS[ mon - 1 ] year hour min sec",
   "question": "What do a string represent in seconds in seconds ?",
   "answer": "time"
  },
  {
   "id": 8414,
   "code": "def allow lazy func *resultclasses def wrapper *args **kwargs for arg in list args + kwargs values if isinstance arg Promise breakelse return func *args **kwargs return lazy func *resultclasses *args **kwargs return wraps func wrapper",
   "question": "How be a function called ?",
   "answer": "with one or more lazy arguments"
  },
  {
   "id": 43237,
   "code": "def TIMEOUT v context defaults['timeout'] int v",
   "question": "How does a timeout for tube operations set ?",
   "answer": "via context"
  },
  {
   "id": 31148,
   "code": "def require instance exists using uuid f @functools wraps f def wrapper context instance uuid *args **kwargs instance get by uuid context instance uuid return f context instance uuid *args **kwargs return wrapper",
   "question": "What requires the specified instance to exist ?",
   "answer": "decorator"
  },
  {
   "id": 36031,
   "code": "def base 64 decode input errors 'strict' assert errors 'strict' output base 64 decodestring input return output len input",
   "question": "What does the code decode ?",
   "answer": "the object input"
  },
  {
   "id": 10821,
   "code": "def get Element Nodes By Key element Node key if key not in element Node attributes return []word str element Node attributes[key] strip evaluated Link Value get Evaluated Link Value element Node word if evaluated Link Value class name ' Element Node' return [evaluated Link Value]if evaluated Link Value class list return evaluated Link Valueprint ' Warning couldnotget Element Nodesinget Element Nodes By Keyinevaluatefor 'print keyprint evaluated Link Valueprint element Nodereturn []",
   "question": "How does the code get the xml elements ?",
   "answer": "by key"
  },
  {
   "id": 17370,
   "code": "def assert no element by context by value wait time MAX WAIT FOR UNEXPECTED ELEMENT try Web Driver Wait context browser wait time until EC presence of element located by value except Timeout Exception return Trueraise KA Lite Timeout",
   "question": "What does the code raise if the element is * still * found after wait_time seconds ?",
   "answer": "a timeoutexception"
  },
  {
   "id": 34581,
   "code": "@pytest mark skipif 'not HAS BEAUTIFUL SOUP' def test htmlsplitter splitter html HTML Splitter lines [html Soup String Beautiful Soup '<table><tr><th> Col 1 </th><th> Col 2 </th></tr></table>' tr html Soup String Beautiful Soup '<table><tr><td> Data 1 </td><td> Data 2 </td></tr></table>' tr ]expected data [[' Col 1 ' ' Col 2 '] [' Data 1 ' ' Data 2 ']]assert list splitter lines expected data lines append '<tr><td> Data 3 </td><td> Data 4 </td></tr>' with pytest raises Type Error list splitter lines with pytest raises core Inconsistent Table Error list splitter []",
   "question": "How do htmlsplitter input lines of type soupstring to return a generator that gives all header and data elements ?",
   "answer": "correctly"
  },
  {
   "id": 39753,
   "code": "def build whitespace split regex text def build parts text lexer shlex shlex text lexer whitespace split Truelexer commenters ''if \"'\" in text lexer quotes '\"'elif '\"' in text lexer quotes \"'\"return list lexer regex ''for line in text splitlines parts [re escape s for s in build parts line ]regex + ' ? [\\\\s]+ ?{ 0 } ? [\\\\s]+ ?' format ' ? [\\\\s]+ ?' join parts return ' ?m ^{ 0 }$' format regex",
   "question": "When does the code create a regular expression ?",
   "answer": "at runtime"
  },
  {
   "id": 27730,
   "code": "def get processor format mapping try obj info mapping[format]except Key Error if format is None raise Value Error ' Formatrequired lowercasestring ' elif not isinstance format basestring raise Type Error ' Needastringforthefileformat lowercase ' elif format format lower raise Value Error ' Formatstring%rshouldbelowercase' % format else raise Value Error ' Unknownformat%r Supportedformatsare%r' % format \"' '\" join mapping mod name obj name obj infomod import ' Bio Search IO %s' % mod name fromlist [''] return getattr mod obj name",
   "question": "What processs the given format ?",
   "answer": "the object"
  },
  {
   "id": 13880,
   "code": "def pairwise most common sets from sympy utilities iterables import subsetsfrom collections import defaultdictmost -1 for i j in subsets list range len sets 2 com sets[i] & sets[j] if com and len com > most best defaultdict list best keys []most len com if len com most if com not in best keys best keys append com best[best keys index com ] append i j if most -1 return []for k in range len best best keys[k] best keys[k] best[k] best keys sort key lambda x len x[ 1 ] return best keys",
   "question": "Where is a list of tuples giving the indices of the pairs of sets in which those elements appeared ?",
   "answer": "where"
  },
  {
   "id": 35343,
   "code": "def check db locks func def new func app *args **kwargs retval func app *args **kwargs temp session app session factory temp session execute 'CREATETABL Edummy foo INT ' temp session execute 'DROPTABL Edummy' temp session close return retvalreturn new func",
   "question": "When are no locks held on the applications database by creating and dropping a dummy table ?",
   "answer": "upon exit"
  },
  {
   "id": 49497,
   "code": "def macro name filename source **identifiers if not identifiers has key 'name' identifiers['name'] namesource source % identifiers codeplace '<%s macro >' % filename code compile source codeplace 'exec' sm sys modulestprm 'twisted python reflect macros'if not sm has key tprm macros new module tprm sm[tprm] macrosmacros count 0macros sm[tprm]macros count + 1macroname 'macro ' + str macros count tprmm tprm + ' ' + macroname mymod new module tprmm sys modules[tprmm] mymodsetattr macros macroname mymod dict mymod dict exec code in dict dictreturn dict[name]",
   "question": "What allows you to create macro - like behaviors in python macro ?",
   "answer": "this"
  },
  {
   "id": 47836,
   "code": "def resolve prop prop if hasattr prop ' proxied property' return prop proxied propertyreturn prop",
   "question": "What does the code resolve ?",
   "answer": "proxied property"
  },
  {
   "id": 45211,
   "code": "def token callable token local dict global dict next Token None func local dict get token[ 1 ] if not func func global dict get token[ 1 ] return callable func and not isinstance func sympy Symbol",
   "question": "What represents a callable function ?",
   "answer": "a token name"
  },
  {
   "id": 13182,
   "code": "def attach role policy policy name role name region None key None keyid None profile None conn get conn region region key key keyid keyid profile profile policy arn get policy arn policy name region key keyid profile try conn attach role policy policy arn role name log info ' Attached{ 0 }policytorole{ 1 } ' format policy name role name except boto exception Boto Server Error as e log debug e msg ' Failedtoattach{ 0 }policytorole{ 1 } 'log error msg format policy name role name return Falsereturn True",
   "question": "What does the code attach to a role ?",
   "answer": "a managed policy"
  },
  {
   "id": 46723,
   "code": "def slave utilization registry xml parent data utilization XML Sub Element xml parent 'com suryagaddipati jenkins Slave Utilization Property' percent int data get 'slave-percentage' 0 XML Sub Element utilization 'needs Exclusive Access To Node' text 'true' if percent else 'false' XML Sub Element utilization 'slave Utilization Percentage' text str percent XML Sub Element utilization 'single Instance Per Slave' text str data get 'single-instance-per-slave' False lower",
   "question": "What wants to use ?",
   "answer": "a job"
  },
  {
   "id": 20428,
   "code": "def external program check to check frozenset [PSQL BIN LZOP BIN PV BIN] could not run []error msgs []def psql err handler popen assert popen returncode 0 error msgs append textwrap fill ' Couldnotgetaconnectiontothedatabase notethatsuperuseraccessisrequired' raise Environment Error 'INTERNAL Hadproblemsrunningpsqlfromexternal program check' with open os devnull 'wb' as nullf for program in to check try if program is PSQL BIN psql csv run 'SELECT 1 ' error handler psql err handler else if program is PV BIN extra args ['--quiet']else extra args []proc popen sp [program] + extra args stdout nullf stderr nullf stdin subprocess PIPE proc stdin close proc wait except Environment Error could not run append program if could not run error msgs append ' Couldnotrunthefollowingprograms aretheyinstalled?' + ' ' join could not run if error msgs raise User Exception 'couldnotrunoneormoreexternalprograms WAL- Edependsupon' '\\n' join error msgs return None",
   "question": "When do confusing error output get ?",
   "answer": "when one does not install a dependency because of the fork - worker model that is both necessary for throughput and makes more obscure the cause of failures"
  },
  {
   "id": 26148,
   "code": "def exists key host None port None db None password None server connect host port db password return server exists key",
   "question": "What exists in redis ?",
   "answer": "the key"
  },
  {
   "id": 44019,
   "code": "def test freeze with pip script result script pip 'freeze' '--all' assert 'pip ' in result stdout",
   "question": "What shows itself ?",
   "answer": "test pip"
  }
 ],
 "dev": [
  {
   "id": 5238,
   "code": "def remove dir path if sabnzbd WIN 32 retries 15 while retries > 0 try os rmdir path returnexcept Windows Error as err if err[ 0 ] 32 logging debug ' Retrydelete%s' path retries - 1else raise Windows Error err time sleep 3 raise Windows Error err else os rmdir path",
   "question": "How does the code remove directory ?",
   "answer": "with retries for win32"
  },
  {
   "id": 912,
   "code": "def shortest augmenting path G s t capacity 'capacity' residual None value only False two phase False cutoff None R shortest augmenting path impl G s t capacity residual two phase cutoff R graph['algorithm'] 'shortest augmenting path'return R",
   "question": "What do a maximum single - commodity flow use ?",
   "answer": "the shortest augmenting path algorithm"
  },
  {
   "id": 204,
   "code": "def parse mime message mime message if isinstance mime message email Message Message return mime messageelif isinstance mime message basestring return email message from string mime message else return email message from file mime message",
   "question": "What does helper function convert to email in ?",
   "answer": "a mime_message"
  },
  {
   "id": 6074,
   "code": "def is ignorable 404 uri if getattr settings 'IGNORABLE 404 STARTS' import warningswarnings warn ' The IGNORABLE 404 START Ssettinghasbeendeprecatedinfavorof IGNORABLE 404 URLS ' Deprecation Warning for start in settings IGNORABLE 404 STARTS if uri startswith start return Trueif getattr settings 'IGNORABLE 404 ENDS' import warningswarnings warn ' The IGNORABLE 404 END Ssettinghasbeendeprecatedinfavorof IGNORABLE 404 URLS ' Deprecation Warning for end in settings IGNORABLE 404 ENDS if uri endswith end return Truereturn any pattern search uri for pattern in settings IGNORABLE 404 URLS",
   "question": "Should a 404 at the given url notify the site managers ?",
   "answer": "No"
  },
  {
   "id": 2253,
   "code": "def Get Calling Module return Get Calling Module Object And Name [1 ]",
   "question": "What does the code return ?",
   "answer": "the name of the module that s calling into this module"
  },
  {
   "id": 2006,
   "code": "def get more info for log more info []site getattr frappe local u'site' None if site more info append u' Site {0 }' format site form dict getattr frappe local u'form dict' None if form dict more info append u' Form Dict {0 }' format frappe as json form dict if more info more info more info + [u''] return u'\\n' join more info",
   "question": "What does the code add ?",
   "answer": "site"
  },
  {
   "id": 1828,
   "code": "def test override column class names class My Table tables Table population tables Column verbose name u' Population' def get column class names self classes set bound column classes set add u'prefix-%s' % bound column name return classes set TEST DATA [{u'name' u' Belgium' u'population' 11200000 } {u'name' u' Luxembourgh' u'population' 540000 } {u'name' u' France' u'population' 66000000 }]html My Table TEST DATA as html build request assert u'<tdclass \"prefix-population\"> 11200000 </td>' in html",
   "question": "How do we control the output of css class names for a column ?",
   "answer": "by overriding get_column_class_names"
  },
  {
   "id": 1143,
   "code": "def read handle record read handle remainder handle read if remainder raise Value Error ' Morethanone Prositerecordfound' return record",
   "question": "What did the code read ?",
   "answer": "one prosite record"
  },
  {
   "id": 6033,
   "code": "def rowadd matlist index 1 index 2 k K result []for i in range len matlist[index 1 ] matlist[index 1 ][i] matlist[index 1 ][i] + k * matlist[index 2 ][i] return matlist",
   "question": "What does the code add ?",
   "answer": "the index1 row with index2 row which in turn is multiplied by k"
  },
  {
   "id": 839,
   "code": "def assert python ok *args **env vars return assert python True *args **env vars",
   "question": "How do the interpreter run ?",
   "answer": "with args and optional environment variables"
  },
  {
   "id": 5543,
   "code": "def get info certificate id returncertificate False returntype None opts salt utils namecheap get opts 'namecheap ssl getinfo' opts['certificate ID'] certificate idif returncertificate opts['returncertificate'] 'true'if returntype is None salt utils namecheap log error 'returntypemustbespecifiedwhenreturncertificateissetto True' raise Exception 'returntypemustbespecifiedwhenreturncertificateissetto True' if returntype not in [' Individual' 'PKCS 7 '] salt utils namecheap log error 'returntypemustbespecifiedas Individualor PKCS 7 not' + returntype raise Exception 'returntypemustbespecifiedas Individualor PKCS 7 not' + returntype opts['returntype'] returntyperesponse xml salt utils namecheap get request opts if response xml is None return {}sslinforesult response xml get Elements By Tag Name 'SSL Get Info Result' [0 ]return salt utils namecheap xml to dict sslinforesult",
   "question": "What does the ssl certificate with two keys \" ssl \" contain ?",
   "answer": "the metadata information \" certificate \" contains the"
  },
  {
   "id": 6067,
   "code": "def get Length Minus One Minimum One element List return max 1 len element List - 1",
   "question": "What does the code get ?",
   "answer": "the length of the length minus one"
  },
  {
   "id": 4467,
   "code": "def expose api raw anonymous func return expose api func to json False user required False",
   "question": "Does the code dump the results to json ?",
   "answer": "No"
  },
  {
   "id": 712,
   "code": "@step ' Icreatealogrecordwith' def step I create logrecord with table context assert context table 'REQUIRE context table'assert len context table rows 1 'REQUIRE table row size 1'step I create logrecords with table context",
   "question": "What do a table use ?",
   "answer": "to provide the parts"
  },
  {
   "id": 4837,
   "code": "def load apps app blacklist global DESKTOP MODULE Sglobal DESKTOP APP Sif DESKTOP APPS is not None raise Exception 'load appshasalreadybeencalled ' DESKTOP APPS []for sdk app in pkg resources iter entry points 'desktop sdk application' if sdk app name not in app blacklist if 'oozie' in app blacklist and sdk app name in 'pig' 'jobsub' LOG warn '%sdependsonooziewhichisblacklisted willskiploading%sapp ' % sdk app name sdk app name else m sdk app load dmi Desktop Module Info m DESKTOP APPS append dmi LOG debug ' Loaded Desktop Applications ' + ' ' join a name for a in DESKTOP APPS DESKTOP MODULES + DESKTOP APPS",
   "question": "What does the code load ?",
   "answer": "the applications from the directories in app_dirs"
  },
  {
   "id": 3456,
   "code": "def pem b name s1 b2 a base 64 b [ -1 ]s 2 ''while s1 s2 + s1 [ 64 ] + '\\n' s1 s1 [ 64 ]s '-----BEGIN%s-----\\n' % name + s2 + '-----END%s-----\\n' % name return s",
   "question": "What does the code encode ?",
   "answer": "a payload bytearray"
  },
  {
   "id": 260,
   "code": "def get bin seeds X bin size min bin freq 1 bin sizes defaultdict int for point in X binned point np round point / bin size bin sizes[tuple binned point ] + 1bin seeds np array [point for point freq in six iteritems bin sizes if freq > min bin freq ] dtype np float 32 if len bin seeds len X warnings warn ' Binningdatafailedwithprovidedbin size %f usingdatapointsasseeds ' % bin size return Xbin seeds bin seeds * bin size return bin seeds",
   "question": "What finds for mean_shift ?",
   "answer": "seeds"
  },
  {
   "id": 244,
   "code": "@help Bar Builder def Bar data label None values None color None stack None group None agg 'sum' xscale 'categorical' yscale 'linear' xgrid False ygrid True continuous range None **kw if continuous range and not isinstance continuous range Range 1 d raise Value Error 'continuous rangemustbeaninstanceofbokeh models ranges Range 1 d' if label is not None and values is None kw['label only'] Trueif agg 'sum' or agg 'mean' agg 'count'values labely range continuous rangekw['label'] labelkw['values'] valueskw['color'] colorkw['stack'] stackkw['group'] groupkw['agg'] aggkw['xscale'] xscalekw['yscale'] yscalekw['xgrid'] xgridkw['ygrid'] ygridkw['y range'] y rangechart create and build Bar Builder data **kw if len chart x range factors 1 and not label chart below[ 0 ] visible Falsereturn chart",
   "question": "What does the code create using : class : barbuilder < bokeh ?",
   "answer": "a bar chart"
  },
  {
   "id": 767,
   "code": "def bad request req resp **kwargs raise HTTP Bad Request ' Badrequest' ' Invalid HTT Pmethod'",
   "question": "What does the code raise ?",
   "answer": "400 httpbadrequest error"
  },
  {
   "id": 1791,
   "code": "def button action button action connect button button action trigger",
   "question": "What does the code make ?",
   "answer": "a button trigger an action"
  },
  {
   "id": 1905,
   "code": "def progressbar iterable None length None label None show eta True show percent None show pos False item show func None fill char '#' empty char '-' bar template '% label s[% bar s]% info s' info sep '' width 36 file None color None from termui impl import Progress Barcolor resolve color default color return Progress Bar iterable iterable length length show eta show eta show percent show percent show pos show pos item show func item show func fill char fill char empty char empty char bar template bar template info sep info sep file file label label width width color color",
   "question": "When did over something iterate ?",
   "answer": "while showing a progress bar"
  },
  {
   "id": 4139,
   "code": "def color Str c return '% 02 x' * 4 % color Tuple c",
   "question": "What does the code generate from a qcolor ?",
   "answer": "a hex string code"
  },
  {
   "id": 4931,
   "code": "def Sample Rows df nrows replace False indices np random choice df index nrows replace replace sample df loc[indices]return sample",
   "question": "What does the code choose from a dataframe ?",
   "answer": "a sample of rows"
  },
  {
   "id": 217,
   "code": "def extract gcs tokens full object name bucket name Noneobject name Nonetokens full object name split '/' if len tokens < 3 logging error \" Malformed GC Spath'{ 0 }' Aborting GC Soperation \" format full object name return bucket name object name bucket name tokens[ 2 ]object name ''for token in tokens[ 3 -1 ] object name + token + '/' object name + tokens[ -1 ]return bucket name object name",
   "question": "Does the code extract the bucket and object name from a full gcs path ?",
   "answer": "Yes"
  },
  {
   "id": 4597,
   "code": "def get view to display matrix scene from mayavi core ui mayavi scene import Mayavi Scenefrom tvtk pyface tvtk scene import TVTK Sceneif not isinstance scene Mayavi Scene TVTK Scene raise Type Error 'scenemustbeaninstanceof TVTK Scene/ Mayavi Scene foundtype%s' % type scene x y tuple scene get size view to disp mat np array [[ x / 2 0 0 0 0 0 x / 2 0 ] [0 0 - y / 2 0 0 0 y / 2 0 ] [0 0 0 0 1 0 0 0] [0 0 0 0 0 0 1 0]] return view to disp mat",
   "question": "For what purpose do the 4x4 matrix return ?",
   "answer": "to convert view coordinates to display coordinates"
  },
  {
   "id": 1628,
   "code": "def effective get get None collection None collection get collection default get if collection else None return get or globals get 'get' or collection get",
   "question": "What used in a given situation ?",
   "answer": "the effective get method"
  },
  {
   "id": 5865,
   "code": "def get next double context builder state ptr a builder lshr get next int 32 context builder state ptr const int 5 b builder lshr get next int 32 context builder state ptr const int 6 a builder uitofp a double b builder uitofp b double return builder fdiv builder fadd b builder fmul a ir Constant double 67108864 0 ir Constant double 9007199254740992 0",
   "question": "When do by the prng generate the next double ?",
   "answer": "at * state_ptr *"
  },
  {
   "id": 5323,
   "code": "def declarative constructor self **kwargs cls type self for k in kwargs if not hasattr cls k raise Type Error '%risaninvalidkeywordargumentfor%s' % k cls name setattr self k kwargs[k]",
   "question": "What does a simple constructor allow ?",
   "answer": "initialization from kwargs"
  },
  {
   "id": 5745,
   "code": "def dict keys startswith dictionary string matched keys [key for key in dictionary keys if key startswith string ]return dict k dictionary[k] for k in matched keys",
   "question": "What do a dictionary contain ?",
   "answer": "the elements of < dict > whose keys start with < string >"
  },
  {
   "id": 4464,
   "code": "def vis minibatch im blob rois blob labels blob overlaps import matplotlib pyplot as pltfor i in xrange rois blob shape[ 0 ] rois rois blob[i ]im ind rois[ 0 ]roi rois[ 1 ]im im blob[im ind ] transpose 1 2 0 copy im + cfg PIXEL MEAN Sim im[ 2 1 0 ]im im astype np uint 8 cls labels blob[i]plt imshow im print 'class ' cls 'overlap ' overlaps[i]plt gca add patch plt Rectangle roi[ 0 ] roi[ 1 ] roi[ 2 ] - roi[ 0 ] roi[ 3 ] - roi[ 1 ] fill False edgecolor 'r' linewidth 3 plt show",
   "question": "What does the code visualize for debugging ?",
   "answer": "a mini - batch"
  },
  {
   "id": 3436,
   "code": "@profiler trace@memoizeddef tenant quota usages request tenant id None if not tenant id tenant id request user project iddisabled quotas get disabled quotas request usages Quota Usage for quota in get tenant quota data request disabled quotas disabled quotas tenant id tenant id usages add quota quota get tenant compute usages request usages disabled quotas tenant id get tenant network usages request usages disabled quotas tenant id get tenant volume usages request usages disabled quotas tenant id return usages",
   "question": "What does the code construct ?",
   "answer": "our usage object"
  },
  {
   "id": 1805,
   "code": "def write timeseries value client project resource custom metric type instance id metric kind now get now rfc 3339 timeseries data {'metric' {'type' custom metric type 'labels' {'environment' 'STAGING'}} 'resource' {'type' 'gce instance' 'labels' {'instance id' instance id 'zone' 'us-central 1 -f'}} 'points' [{'interval' {'start Time' now 'end Time' now} 'value' {'int 64 Value' get custom data point }}]}request client projects time Series create name project resource body {'time Series' [timeseries data]} request execute",
   "question": "When did by get_custom_data_point obtain the custom metric ?",
   "answer": "at a point in time"
  }
 ],
 "test": [
  {
   "id": 5238,
   "code": "def get featurestate name featurestate featurestate cache get name None if featurestate is None featurestate Feature State name world featurestate cache[name] featurestatereturn featurestate",
   "question": "What does the code get ?",
   "answer": "a featurestate object for this feature"
  },
  {
   "id": 912,
   "code": "def update pathtozip patterns filepaths names compression zipfile ZIP DEFLATED verbose True assert len patterns len filepaths len names z zipfile Zip File pathtozip mode 'a' for name in z namelist for pat fname new name in zip patterns filepaths names if pat search name if verbose print ' Updating%swith%s' % name fname if new name is None z replace fname arcname name compress type compression else z delete name z write fname new name compress type compression breakz close",
   "question": "Where do files in the zip file update matching the given patterns ?",
   "answer": "at pathtozip"
  },
  {
   "id": 204,
   "code": "def saferepr obj repr py io saferepr obj if py builtin istext repr t py builtin textelse t py builtin bytesreturn repr replace t '\\n' t '\\\\n'",
   "question": "Does the code get a safe repr of an object for assertion error messages ?",
   "answer": "Yes"
  },
  {
   "id": 6074,
   "code": "def load module filename import magics marshalfp open filename 'rb' magic fp read 4 try version magics versions[magic]except Key Error raise Import Error ' Unknownmagicnumber%sin%s' % ord magic[ 0 ] + 256 * ord magic[ 1 ] filename if version '2 7' raise Import Error ' Thisisa Python%sfile Only Python 2 7filesaresupported ' % version fp read 4 co marshal load fp fp close return version co",
   "question": "What do file contain ?",
   "answer": "python byte - code object code_object : code_object"
  },
  {
   "id": 2253,
   "code": "@app after requestdef after response print 'andherewehavetheresponseobjectinstead ' response return response",
   "question": "What must take one parameter ?",
   "answer": "your function"
  },
  {
   "id": 2006,
   "code": "def set Unjellyable Factory For Class classname copy Factory global unjellyable Factory Registryclassname maybe Class classname unjellyable Factory Registry[classname] copy Factoryglobal Security allow Types classname",
   "question": "What did the factory construct ?",
   "answer": "a remote instance of a type : : jellier"
  },
  {
   "id": 1828,
   "code": "def format issue issue ret {'id' issue get 'id' 'issue number' issue get 'number' 'state' issue get 'state' 'title' issue get 'title' 'user' issue get 'user' get 'login' 'html url' issue get 'html url' }assignee issue get 'assignee' if assignee assignee assignee get 'login' labels issue get 'labels' label names []for label in labels label names append label get 'name' milestone issue get 'milestone' if milestone milestone milestone get 'title' ret['assignee'] assigneeret['labels'] label namesret['milestone'] milestonereturn ret",
   "question": "What formats into a more manageable and useful dictionary for issue information ?",
   "answer": "api return information"
  },
  {
   "id": 1143,
   "code": "def clone via serialize obj s c Pickle dumps obj get pickle protocol return c Pickle loads s",
   "question": "What does the code make by serializing it and then deserializing it ?",
   "answer": "a \" deep copy \" of an object"
  },
  {
   "id": 6033,
   "code": "def compare Function Name first second first get Converted Name first second get Converted Name second if first < second return -1 return first < second",
   "question": "What does the code compare ?",
   "answer": "the function names"
  },
  {
   "id": 839,
   "code": "def serve webapp webapp port None host None server Noneif port is not None server httpserver serve webapp host host port port start loop False else random seed for i in range 0 9 try port str random randint 8000 10000 server httpserver serve webapp host host port port start loop False breakexcept socket error as e if e[ 0 ] 98 continueraiseelse raise Exception ' Unabletoopenaportbetween%sand%stostart Galaxyserver' % 8000 1000 t threading Thread target server serve forever t start return server port",
   "question": "What does the code serve on a recommend port or a free one ?",
   "answer": "the webapp"
  },
  {
   "id": 5543,
   "code": "def Validate Attributes tag attributes goodattributes all good Truefor attr in attributes keys if not attr in goodattributes output Error ' Unknown%sattribute %s' % tag attr all good Falsereturn all good",
   "question": "What does attributes not contain ?",
   "answer": "any attribute not listed in goodattributes"
  },
  {
   "id": 6067,
   "code": "def Zig Zag Encode value if value > 0 return value << 1 return value << 1 ^ ~ 0",
   "question": "How can they be used ?",
   "answer": "effectively"
  },
  {
   "id": 4467,
   "code": "def restart app id restart False force True ret {'restarted' None}if not restart ret['restarted'] Falsereturn rettry response salt utils http query '{ 0 }/v 2 /apps/{ 1 }/restart?force {2 }' format base url app id id force method 'POST' decode type 'json' decode True header dict {' Content- Type' 'application/json' ' Accept' 'application/json'} log debug 'restartresponse %s' response ret['restarted'] Trueret update response['dict'] return retexcept Exception as ex log error 'unabletorestartmarathonapp %s' ex message return {'exception' {'message' ex message}}",
   "question": "What does the code restart ?",
   "answer": "the current server configuration for the specified app"
  },
  {
   "id": 712,
   "code": "def close *args if len args 0 fig Manager pylab helpers Gcf get active if fig Manager is None returnelse fig Manager canvas mpl disconnect fig Manager cidgcf pylab helpers Gcf destroy fig Manager num elif len args 1 arg args[ 0 ]if arg 'all' for manager in pylab helpers Gcf get all fig managers manager canvas mpl disconnect manager cidgcf pylab helpers Gcf destroy manager num elif isinstance arg int pylab helpers Gcf destroy arg elif isinstance arg Figure for manager in pylab helpers Gcf get all fig managers if manager canvas figure arg manager canvas mpl disconnect manager cidgcf pylab helpers Gcf destroy manager num else raise Type Error ' Unrecognizedargumenttype%stoclose' % type arg else raise Type Error 'closetakes 0 or 1 arguments'",
   "question": "What is a : class : figure instance where ?",
   "answer": "h"
  },
  {
   "id": 4837,
   "code": "def IQR xs cdf Cdf xs return cdf Value 0 25 cdf Value 0 75",
   "question": "What does the code compute ?",
   "answer": "the interquartile of a sequence"
  },
  {
   "id": 3456,
   "code": "def get non generated file lines lines to copy []flag found Falsewith open ' /plotly/graph objs/graph objs py' 'r' as f for line to copy in f if line to copy startswith FLAG flag found Truebreaklines to copy append line to copy if not flag found raise Value Error ' Failedtofindflag \\n\"{}\"\\ningraph objs tools py ' format FLAG return lines to copy",
   "question": "What copys to our special flag line ?",
   "answer": "each line up"
  },
  {
   "id": 260,
   "code": "def find checks argument name checks []function type type find checks for name function in globals iteritems if type function is function type args inspect getargspec function [0 ]if len args > 1 and args[ 0 ] startswith argument name checks append name function args checks sort return checks",
   "question": "What does the code find ?",
   "answer": "all globally visible functions where the first argument name starts with argument_name"
  },
  {
   "id": 244,
   "code": "def memoize stacks func cache {}@wraps func def memoizer be optree optree key tensor index map index tensor map optree intrinsic key maps optree key optree key id be if optree key in cache stacks cached tensor index map cache[optree key]for stack in stacks for i in range len stack if isinstance stack[i] Tensor if stack[i] in cached tensor index map stack[i] index tensor map[cached tensor index map[stack[i]]]cache[optree key] stacks tensor index map else stacks func be optree cache[optree key] stacks tensor index map return stacksreturn memoizer",
   "question": "What does the code use ?",
   "answer": "intrinsic_key_maps"
  },
  {
   "id": 767,
   "code": "def feature permission url option msg yes action no action abort on config val config get *option if config val 'ask' if url is Valid text ' Allowthewebsiteat<b>{}</b>to{}?' format html escape url to Display String msg else text ' Allowthewebsiteto{}?' format msg return message confirm async yes action yes action no action no action cancel action no action abort on abort on title ' Permissionrequest' text text elif config val yes action return Noneelse no action return None",
   "question": "What does the code handle ?",
   "answer": "a feature permission request"
  },
  {
   "id": 1791,
   "code": "def deterministic Pool worker doer create Memory Worker return Deterministic Thread Pool Team Lock Worker Lock local lambda worker lambda None doer",
   "question": "What does the code create ?",
   "answer": "a deterministic threadpool"
  },
  {
   "id": 1905,
   "code": "def Int 2 AP num val a2 p [] 'ABCDEFGHIJKLMNOP' num int abs num while num num mod divmod num 16 val insert 0 a2 p[mod] return '' join val",
   "question": "How did the code string = int2ap return convert to a string ?",
   "answer": "using characters from the set a"
  },
  {
   "id": 4139,
   "code": "def decode preferred encoding s enc locale getpreferredencoding try try return s decode enc except Lookup Error enc DEFAULT ENCODIN Greturn s decode enc except Unicode Decode Error return s decode u'latin- 1 '",
   "question": "What does the code use ?",
   "answer": "the preferred encoding for the locale or"
  },
  {
   "id": 4931,
   "code": "def configure logging config format config get 'log format' '% name s% levelname s% asctime s% message s' level logging level Names[config get 'log level' 'DEBUG' ]destination config get 'log destination' 'stdout' log info \" Loggingat'%s'levelto'%s'\" % level destination root logging get Logger root set Level level if level < logging DEBUG logging get Logger 'paste httpserver Thread Pool' set Level logging WARN for h in root handlers[ ] root remove Handler h if destination 'stdout' handler logging Stream Handler sys stdout else handler logging File Handler destination formatter logging Formatter format handler set Formatter formatter root add Handler handler",
   "question": "What does the code allow ?",
   "answer": "some basic logging configuration to be read from the cherrpy config"
  },
  {
   "id": 217,
   "code": "def expand empty tensor var size if size 0 return tensor varshapes [tensor var shape[x] for x in xrange tensor var ndim ]new shape [ size + shapes[ 0 ] ] + shapes[ 1 ] empty tensor Alloc Empty tensor var dtype *new shape ret tensor set subtensor empty[ shapes[ 0 ]] tensor var ret tag nan guard mode check Falsereturn ret",
   "question": "Where do uninitialized memory add ?",
   "answer": "at the end of the tensor"
  },
  {
   "id": 4597,
   "code": "@validate 'form' 'graph' 'tree' def valid att in field arch return not arch xpath '//field[not @name ]'",
   "question": "What must field nodes have all ?",
   "answer": "a @name"
  },
  {
   "id": 1628,
   "code": "def O Auth Authorize Token CGI method parameters outfile oauth callback Get First parameters OAUTH CALLBACK PARAM '' if method 'GET' outfile write ' Status 200 \\r\\n' outfile write ' Content- Type text/html\\r\\n' outfile write '\\r\\n' outfile write Render Token Approval Template oauth callback elif method 'POST' if oauth callback outfile write ' Status 302 Redirectingtocallback URL\\r\\n' outfile write ' Location %s\\r\\n' % oauth callback outfile write '\\r\\n' else outfile write ' Status 200 \\r\\n' outfile write ' Content- Type text/html\\r\\n' outfile write '\\r\\n' outfile write Render Token Approved Template else outfile write ' Status 400 Unsupportedmethod\\r\\n'",
   "question": "What does the code run ?",
   "answer": "the oauthauthorizetoken cgi"
  },
  {
   "id": 5865,
   "code": "def md return Mde 2 md Converter",
   "question": "What does this make ?",
   "answer": "a converter from markdown - extra to markdown"
  },
  {
   "id": 5323,
   "code": "def read png filename x Reader filename try alpha x as Direct [3 ]['alpha']if alpha y x as RGBA 8 [2 ]n 4else y x as RGB 8 [2 ]n 3y np array [yy for yy in y] np uint 8 finally x file close y shape y shape[ 0 ] y shape[ 1 ] // n n return y",
   "question": "By how much did the code read a png file unlike imread ?",
   "answer": "to rgb8 or rgba8"
  },
  {
   "id": 5745,
   "code": "def make assert error def openssl assert ok '\\n If*ok*isnot True retrievetheerrorfrom Open SS Landraiseit \\n'if ok is not True exception from error queue error return openssl assert",
   "question": "What does the code create ?",
   "answer": "an assert function that uses : func : exception_from_error_queue to raise an exception wrapped by * error *"
  },
  {
   "id": 4464,
   "code": "def delivery pipeline registry xml parent data pvc XML Sub Element xml parent 'se diabol jenkins pipeline Pipeline Version Contributor' pvc set 'plugin' 'delivery-pipeline-plugin' mapping [ 'version-template' 'version Template' '' 'set-display-name' 'update Display Name' False ]convert mapping to xml pvc data mapping fail required True",
   "question": "What will the job create if enabled ?",
   "answer": "a version based on the template"
  },
  {
   "id": 3436,
   "code": "def config read reseller options conf defaults reseller prefix opt conf get 'reseller prefix' 'AUTH' split ' ' reseller prefixes []for prefix in [pre strip for pre in reseller prefix opt if pre strip ] if prefix \"''\" prefix ''prefix append underscore prefix if prefix not in reseller prefixes reseller prefixes append prefix if len reseller prefixes 0 reseller prefixes append '' associated options {}for prefix in reseller prefixes associated options[prefix] dict defaults associated options[prefix] update config read prefixed options conf '' defaults prefix name prefix if prefix '' else \"''\" associated options[prefix] update config read prefixed options conf prefix name defaults return reseller prefixes associated options",
   "question": "In which direction do reseller_prefix option and associated options read ?",
   "answer": "from configuration"
  },
  {
   "id": 1805,
   "code": "def get custom metric client project id custom metric type request client projects metric Descriptors list name project id filter 'metric type starts with \"{}\" ' format custom metric type response request execute print ' List Custom Metricsresponse 'pprint pprint response try return response['metric Descriptors']except Key Error return None",
   "question": "What does the code retrieve ?",
   "answer": "the custom metric we created"
  }
 ]
}

In [3]:
assert {split: len(rows) for split, rows in MINI_DATA.items()} == {"train":256,"dev":32,"test":32}
for split, rows in MINI_DATA.items():
    assert len({r["id"] for r in rows}) == len(rows)
    assert all(r["code"] and r["question"] and r["answer"] for r in rows)
    print(split, len(rows))
example = MINI_DATA["train"][0]
print("\nQuestion:", example["question"], "\nAnswer:", example["answer"])

train 256
dev 32
test 32

Question: What does the code append to the list of saved errors ? 
Answer: an error message


## 2. Tokenizer sederhana dan tensor

Tokenizer berbasis kata/tanda baca, dengan vocabulary kecil yang dipelajari dari train.
Kata baru menjadi `<unk>`. Ini berbeda dari tokenizer CodeBERT.
Pertanyaan diletakkan sebelum kode, dipisahkan token khusus.

In [4]:
PAD, BOS, EOS, UNK, SEP = range(5)
SPECIAL = ["<pad>", "<bos>", "<eos>", "<unk>", "<sep>"]
def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

counts = Counter(token for row in MINI_DATA["train"]
                 for field in ("question", "code", "answer") for token in tokenize(row[field]))
vocab = SPECIAL + [token for token, _ in counts.most_common(VOCAB_LIMIT-len(SPECIAL)) if token not in SPECIAL]
stoi = {token: i for i, token in enumerate(vocab)}
def encode(tokens):
    return [stoi.get(token, UNK) for token in tokens]
def tensors(rows):
    sources, lengths, targets = [], [], []
    for row in rows:
        source = (encode(tokenize(row["question"])) + [SEP] + encode(tokenize(row["code"])))[:MAX_SOURCE]
        target = [BOS] + encode(tokenize(row["answer"]))[:MAX_TARGET-2] + [EOS]
        lengths.append(len(source))
        sources.append(source + [PAD]*(MAX_SOURCE-len(source)))
        targets.append(target + [PAD]*(MAX_TARGET-len(target)))
    return TensorDataset(torch.tensor(sources), torch.tensor(lengths), torch.tensor(targets))

datasets = {split:tensors(rows) for split, rows in MINI_DATA.items()}
assert datasets["train"].tensors[0].shape == (256, MAX_SOURCE)
assert all((d.tensors[1] > 0).all() for d in datasets.values())
assert tokenize("Hello, CODE!") == ["hello", ",", "code", "!"]
assert encode(["word_not_present_in_this_vocab_98375"]) == [UNK]
print("Vocabulary:", len(vocab), "| Train tensor:", tuple(datasets["train"].tensors[0].shape))

Vocabulary: 2000 | Train tensor: (256, 96)


## 3. Model mini

GRU encoder membaca pertanyaan dan kode. GRU decoder menghasilkan jawaban satu token demi satu token.
Packed sequence membuat encoder mengabaikan padding. Training memakai teacher forcing; inferensi memakai prediksi model sendiri.

In [5]:
class MiniCodeQA(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), EMBED_SIZE, padding_idx=PAD)
        self.encoder = nn.GRU(EMBED_SIZE, HIDDEN_SIZE, batch_first=True)
        self.decoder = nn.GRU(EMBED_SIZE, HIDDEN_SIZE, batch_first=True)
        self.output = nn.Linear(HIDDEN_SIZE, len(vocab))

    def encode(self, source, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(source), lengths.cpu(),
                                                  batch_first=True, enforce_sorted=False)
        _, hidden = self.encoder(packed)
        return hidden

    def forward(self, source, lengths, decoder_input):
        hidden = self.encode(source, lengths)
        decoded, _ = self.decoder(self.embedding(decoder_input), hidden)
        return self.output(decoded)

    @torch.no_grad()
    def generate(self, source, lengths):
        self.eval()
        hidden = self.encode(source, lengths)
        token = torch.full((source.size(0), 1), BOS, dtype=torch.long, device=source.device)
        done = torch.zeros(source.size(0), dtype=torch.bool, device=source.device)
        generated = []
        for _ in range(MAX_TARGET-1):
            decoded, hidden = self.decoder(self.embedding(token), hidden)
            logits = self.output(decoded[:, -1])
            logits[:, [PAD, BOS, SEP]] = -float("inf")
            token = logits.argmax(-1, keepdim=True)
            token = torch.where(done[:, None], torch.full_like(token, EOS), token)
            generated.append(token)
            done |= token.squeeze(1).eq(EOS)
            if done.all():
                break
        return torch.cat(generated, dim=1)

model = MiniCodeQA().to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
parameter_count = sum(p.numel() for p in model.parameters())
assert parameter_count < 1_000_000
print(f"Parameter: {parameter_count:,} — kurang dari satu juta, tanpa bobot pretrained.")

Parameter: 415,312 — kurang dari satu juta, tanpa bobot pretrained.


## 4. Training singkat dan self-check

Loss validation memilih checkpoint terbaik. Pemeriksaan otomatis memastikan gradient terbentuk,
loss finite, serta loss pada batch uji training turun setelah delapan epoch.

In [6]:
def batch_loss(batch):
    source, lengths, target = [t.to(DEVICE) for t in batch]
    logits = model(source, lengths, target[:, :-1])
    return criterion(logits.reshape(-1, len(vocab)), target[:, 1:].reshape(-1))

@torch.no_grad()
def average_loss(dataset):
    model.eval()
    loss_sum, tokens = 0.0, 0
    for batch in DataLoader(dataset, batch_size=BATCH_SIZE):
        n = int(batch[2][:, 1:].ne(PAD).sum())
        loss_sum += float(batch_loss(batch)) * n
        tokens += n
    return loss_sum/tokens

probe = next(iter(DataLoader(datasets["train"], batch_size=8)))
model.eval()
with torch.no_grad():
    before = float(batch_loss(probe))
model.train()
probe_loss = batch_loss(probe)
probe_loss.backward()
assert model.encoder.weight_ih_l0.grad is not None
assert torch.isfinite(model.encoder.weight_ih_l0.grad).all()
model.zero_grad(set_to_none=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))
best_loss, best_state, history = float("inf"), None, []
train_start = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    summed, tokens = 0.0, 0
    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = batch_loss(batch)
        assert torch.isfinite(loss)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        n = int(batch[2][:, 1:].ne(PAD).sum())
        summed += float(loss.detach()) * n
        tokens += n
    dev_loss = average_loss(datasets["dev"])
    history.append({"epoch":epoch+1,"train_loss":summed/tokens,"dev_loss":dev_loss})
    print(f"Epoch {epoch+1}/{EPOCHS} | train {summed/tokens:.3f} | dev {dev_loss:.3f}")
    if dev_loss < best_loss:
        best_loss = dev_loss
        best_state = copy.deepcopy(model.state_dict())
model.eval()
with torch.no_grad():
    after = float(batch_loss(probe))
assert after < before, (before, after)
model.load_state_dict(best_state)
training_seconds = time.perf_counter()-train_start
print(f"\nTRAINING OK | {training_seconds:.1f} detik | probe loss {before:.3f} menjadi {after:.3f}")

Epoch 1/8 | train 6.672 | dev 4.924


Epoch 2/8 | train 4.864 | dev 4.921


Epoch 3/8 | train 4.543 | dev 4.935


Epoch 4/8 | train 4.298 | dev 4.953


Epoch 5/8 | train 4.047 | dev 4.976


Epoch 6/8 | train 3.794 | dev 5.080


Epoch 7/8 | train 3.533 | dev 5.056


Epoch 8/8 | train 3.256 | dev 5.146

TRAINING OK | 4.1 detik | probe loss 7.568 menjadi 2.926


## 5. Prediksi test dan metrik ringan

EM dan token F1 dihitung pada 32 contoh test yang tidak digunakan memilih checkpoint.
Normalisasi memakai tokenizer mini, sehingga angka ini **tidak sebanding langsung dengan tabel paper**.
Nilai rendah tetap mungkin pada subset kecil dan model dari nol.

In [7]:
def decode(ids):
    result = []
    for index in ids:
        if index == EOS:
            break
        if index not in (PAD, BOS, SEP):
            result.append(vocab[index])
    return " ".join(result)

def answer_score(prediction, reference):
    p, r = tokenize(prediction), tokenize(reference)
    em = float(p == r)
    overlap = sum((Counter(p) & Counter(r)).values())
    f1 = 2*overlap/(len(p)+len(r)) if p or r else 1.0
    return em, f1

assert answer_score("Yes", "yes") == (1.0, 1.0)
assert answer_score("a b", "a c") == (0.0, 0.5)
assert answer_score("", "a") == (0.0, 0.0)
assert answer_score("", "") == (1.0, 1.0)
assert decode([BOS, stoi.get("yes", UNK), EOS, UNK]) == vocab[stoi.get("yes", UNK)]

predictions = []
for source, lengths, _ in DataLoader(datasets["test"], batch_size=BATCH_SIZE):
    output = model.generate(source.to(DEVICE), lengths.to(DEVICE))
    predictions.extend(decode(ids) for ids in output.cpu().tolist())
assert len(predictions) == len(MINI_DATA["test"])
scores = [answer_score(p, row["answer"]) for p, row in zip(predictions, MINI_DATA["test"])]
metrics = {"EM":100*sum(s[0] for s in scores)/len(scores),
           "token_F1":100*sum(s[1] for s in scores)/len(scores)}
assert all(math.isfinite(v) and 0 <= v <= 100 for v in metrics.values())
print("TEST OK:", metrics)
for row, prediction in list(zip(MINI_DATA["test"], predictions))[:5]:
    print("\nQ:", row["question"], "\nGold:", row["answer"], "\nPred:", prediction or "<jawaban kosong>")

TEST OK: {'EM': 0.0, 'token_F1': 10.109126984126984}

Q: What does the code get ? 
Gold: a featurestate object for this feature 
Pred: a

Q: Where do files in the zip file update matching the given patterns ? 
Gold: at pathtozip 
Pred: a

Q: Does the code get a safe repr of an object for assertion error messages ? 
Gold: Yes 
Pred: a

Q: What do file contain ? 
Gold: python byte - code object code_object : code_object 
Pred: a

Q: What must take one parameter ? 
Gold: your function 
Pred: a


## 6. Coba pertanyaan sendiri

Sel ini memakai contoh test pertama. Ganti isi `code_text` dan `question_text` untuk mencoba kode lain.
Model hanya mengenal vocabulary mini; hasil di luar data training dapat tidak masuk akal.

In [8]:
def answer(code_text, question_text):
    row = {"code":code_text,"question":question_text,"answer":"placeholder"}
    source, lengths, _ = tensors([row]).tensors
    return decode(model.generate(source.to(DEVICE), lengths.to(DEVICE))[0].cpu().tolist())

code_text = MINI_DATA["test"][0]["code"]
question_text = MINI_DATA["test"][0]["question"]
print("Code:", code_text)
print("Question:", question_text)
print("Jawaban model:", answer(code_text, question_text) or "<jawaban kosong>")

Code: def get featurestate name featurestate featurestate cache get name None if featurestate is None featurestate Feature State name world featurestate cache[name] featurestatereturn featurestate
Question: What does the code get ?
Jawaban model: a


## 7. Simpan dan tes checkpoint

Checkpoint kecil, prediksi, serta ringkasan hasil disimpan ke `mini_codeqa_results/` di folder notebook.
Self-check terakhir memastikan checkpoint bisa dibuka dan menghasilkan prediksi yang sama.

In [9]:
checkpoint_path = OUTPUT / "mini_codeqa.pt"
torch.save({"model_state":model.state_dict(),"vocab":vocab,
            "config":{"embed_size":EMBED_SIZE,"hidden_size":HIDDEN_SIZE,
                      "max_source":MAX_SOURCE,"max_target":MAX_TARGET}}, checkpoint_path)
state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
assert state["vocab"] == vocab
restored = MiniCodeQA().to(DEVICE)
restored.load_state_dict(state["model_state"], strict=True)
source, lengths, _ = datasets["test"].tensors
assert torch.equal(model.generate(source[:2].to(DEVICE), lengths[:2].to(DEVICE)),
                   restored.generate(source[:2].to(DEVICE), lengths[:2].to(DEVICE)))
elapsed = time.perf_counter()-START
results = {"status":"PASS","purpose":"mini execution demo, not paper reproduction",
           "parameters":parameter_count,"device":str(DEVICE),"epochs":EPOCHS,
           "split_sizes":{s:len(r) for s,r in MINI_DATA.items()},
           "training_seconds":training_seconds,"total_seconds_after_import":elapsed,
           "probe_loss_before":before,"probe_loss_after":after,"metrics":metrics,"history":history}
(OUTPUT / "results.json").write_text(json.dumps(results, indent=2))
with (OUTPUT / "predictions.jsonl").open("w",encoding="utf-8") as f:
    for row, prediction in zip(MINI_DATA["test"],predictions):
        f.write(json.dumps({**row,"prediction":prediction},ensure_ascii=False)+"\n")
print(f"SEMUA TES LULUS | Training {training_seconds:.1f} detik | Total {elapsed:.1f} detik")
print(f"Checkpoint: {checkpoint_path.stat().st_size/2**20:.2f} MiB")
print("Output:", OUTPUT)

SEMUA TES LULUS | Training 4.1 detik | Total 11.4 detik
Checkpoint: 1.62 MiB
Output: /home/inaka/Codes/LAB_IC/CodeQA/mini_codeqa_results
